In [399]:
#Constrain using relative SPE instead of absolute SPE

# > Put in SPE/SRS Dataframe cell

#Put events in order of increasing size from left -> right
SPE_relative_list = [
    ['Annette_Event_2', 'Annette_Event_3', 'Annette_Event_6', 'Annette_Event_4', 'Annette_Event_5'],
    ['Bidart_Event_3', 'Bidart_Event_4'],
    ['Frazier_Event_4', 'Frazier_Event_7', 'Frazier_Event_6', 'Frazier_Event_8', 'Frazier_Event_10', 'Frazier_Event_2', 'Frazier_Event_3', 'Frazier_Event_5'],
    ['Pallett_Event_I', 'Pallett_Event_N', 'Pallett_Event_R', 'Pallett_Event_T', 'Pallett_Event_V'],
    ['Wrightwood_Event_W5', 'Wrightwood_Event_W8', 'Wrightwood_Event_W14', 'Wrightwood_Event_W11', 'Wrightwood_Event_W3', 'Wrightwood_Event_W6', 'Wrightwood_Event_W12', 'Wrightwood_Event_W10', 'Wrightwood_Event_W9', 'Wrightwood_Event_W4']
]

#Put same size events in their own bracketed ([ ]) groups
equal_size_events = [['Frazier_Event_7', 'Frazier_Event_9'], ['Frazier_Event_10', 'Frazier_Event_11'], ['Pallett_Event_C', 'Pallett_Event_D', 'Pallett_Event_F'], ['Wrightwood_Event_W7', 'Wrightwood_Event_W13']]    

# >

# >> Put in ILP Model cell

SPE_relative_constrain = 'on'

equality_threshold = 1 #Maximum allowed difference in lateral slip (in meters) to be allowed to be 'relatively' the same sized event

#Constrain event size compared to other events at that same site
if SPE_relative_constrain == 'on':
    for site_index in range(len(SPE_relative_list)):
        for event_idx in range(len(SPE_relative_list[site_index]) - 1):
            event = SPE_relative_list[site_index][event_index]
            next_event = SPE_relative_list[site_index][event_index + 1]
#Find which candidate events (which 'n') correspond to each trench event and find the displacements matrix that corresponds to the given site
            n_indices = np.where(candidates_df[event] == 1)[0] #Indices of the first event to be compared
            o_indices = np.where(candidates_df[next_event] == 1)[0] #Indices of the next event (larger event, according to arbitrary decision on how dataframe should be ordered) 
            site = event.split('_')[0]
            index = trench_site_names.index(site)
#Create objective functions for all displacement matrices associated with that site and those candidate events
            model += pl.lpSum(trench_displacements[n][index][i] * y[n][i] for n in n_indices for i in range(len(trench_displacements[n][0]))) <= pl.lpSum(trench_displacements[o][index][i] * y[o][i] for o in o_indices for i in range(len(trench_displacements[o][0])))


#Constrain event size to be comparable to other similar-sized events at that same site (within 'equality_threshold' meters, which is arbitrary measurement)

if SPE_relative_constrain == 'on':
    for group in equal_size_events:
        for idx in range(len(group) - 1): #Loop through each pair in the group of similar sized events to constrain displacement difference to be within equality_threshold
#Find which candidate events (which 'n') correspond to each trench event in the pair and find the displacements matrix that corresponds to the given site
            n_indices = [[], []]
#Go through each slip-per-event group and save the indices of the candidate events that each trench event belongs to
            n_indices[0].append(np.where(candidates_df[group[idx]] == 1)[0])
            n_indices[1].append(np.where(candidates_df[group[idx + 1]] == 1)[0])
        #n_indices = [item for sublist in n_indices for item in sublist]
            site = group[0].split('_')[0]
            index = trench_site_names.index(site)
#Create objective functions for all displacements associated with that group at that site
            model += - equality_threshold <= pl.lpSum(trench_displacements[n][index][i] * y[n][i] for n in n_indices[0][0] for i in range(len(trench_displacements[n][0]))) - pl.lpSum(trench_displacements[n][index][i] * y[n][i] for n in n_indices[1][0] for i in range(len(trench_displacements[n][0]))) <= equality_threshold

        ## >>

In [ ]:
##

In [306]:
import numpy as np
import pandas as pd
from itertools import product
from scipy.interpolate import interp1d
import itertools
from itertools import product

##NOTE: PDF Enumearation (Option 1) has not been updated since I optimized CDF Enumeration (Option 2) to be able to handle 8+ trenches
##Do not used PDF Enumeration for more than 5 trenches

Enumeration_Option = 2 #Option 1 = PDF Enumeration, Option 2 = CDF Enumeration
threshold = 0.0003 #Change to set how well event pdfs need to overlap to be included in candidates (default = 0.0003)
prob_threshold = 0.002 #Change to lower/raise the probability threshold of the year of the base of the trench

#Add depositional hiatuses here:
sedimentary_hiatus = {
    'trench': ['Wrightwood'],
    'upper age': [1400],
    'lower age': [1300]
}

Sedimentary_Hiatus_df = pd.DataFrame(sedimentary_hiatus)

#Define sites dataframe for Southern San Andreas Fault
sites = pd.DataFrame({
##Test: Remove new sites, run with old sites to see what is making model infeasible    "site": ["end_n", "Annette", "Bidart", "Frazier", "Elizabeth", "Pallett", "Wrightwood", "Pitman", "Burro", "end_s"],
    #"site": ["end_n", "Bidart", "Frazier", "Elizabeth", "Pallett", "Wrightwood", "end_s"],
    "site": ["end_n", "Annette", "Bidart", "Frazier", "Elizabeth", "Pallett", "Wrightwood", "Pitman", "Burro", "end_s"],
    "loc": [0, 20, 72, 169, 218, 270, 292, 317, 377, 386],
    #"loc": [0, 72, 169, 218, 270, 292, 386],
    "file": [None, "C:/Users/erikc/OneDrive/Documents/Research/Annette_inverted.csv", "C:/Users/erikc/OneDrive/Documents/Research/Bidart.csv", "C:/Users/erikc/OneDrive/Documents/Research/Frazier.csv", "C:/Users/erikc/OneDrive/Documents/Research/Elizabeth_Lake_test.csv", "C:/Users/erikc/OneDrive/Documents/Research/PCreek.csv", "C:/Users/erikc/OneDrive/Documents/Research/Wright.csv", "C:/Users/erikc/OneDrive/Documents/Research/PitmanCanyon.csv", "C:/Users/erikc/OneDrive/Documents/Research/BurroFlats.csv",  None],
    #"file": [None, "C:/Users/erikc/OneDrive/Documents/Research/Bidart.csv", "C:/Users/erikc/OneDrive/Documents/Research/Frazier.csv", "C:/Users/erikc/OneDrive/Documents/Research/Elizabeth_Lake_test.csv", "C:/Users/erikc/OneDrive/Documents/Research/PCreek.csv", "C:/Users/erikc/OneDrive/Documents/Research/Wright.csv",  None],
    "event_code": [None, "Event", "Event", "Event", "Event", "Event", "Event", "Event", "Event", None],
    #"event_code": [None, "Event", "Event", "Event", "Event", "Event", None],
    "earliest": [None, 893, 1350, 601, 1090, 630, 420, 816, -56, None]
    #"earliest": [None, 1350, 601, 1090, 630, 420, None]
})


if Enumeration_Option == 1:
        #Create earthquake class
    class Earthquake:

        #Constructor: stores earthquake name, age, PDF, and CDF for each earthquake event (i.e. Bidart.Event_2)
        def __init__(self, name, age, PDF, CDF):
            self.name = name
            self.age = np.array(age)
            self.PDF = np.array(PDF)
            self.CDF = np.array(CDF)
        #Returns single sampled age based on PDF: generate random number between 0 and 1, then sample the age from the CDF that corresponds to number
        def event(self):
            r = np.random.rand()
            return self.age[np.argmin(np.abs(self.CDF - r))]
        #Returns n sampled ages: calls event() n times, then stores results in an array
        def events(self, n):
            return np.array([self.event() for _ in range(n)])
        #Resamples earthquake age to yearly resolution: calculates age range (L), then creates list of years inside age range and interpolates event PDF and CDF to those years
        def resample(self):
            L = int(self.age[-1] - self.age[0])
            new_age = np.linspace(self.age[0], self.age[-1], L + 1)

            f_age = interp1d(self.age, self.age, fill_value="extrapolate")
            f_pdf = interp1d(self.age, self.PDF, fill_value="extrapolate")
            f_cdf = interp1d(self.age, self.CDF, fill_value="extrapolate")

            self.age = f_age(new_age)
            self.PDF = f_pdf(new_age)
            self.PDF = self.PDF / np.sum(self.PDF)
            self.CDF = f_cdf(new_age)
        #Calculates joint probability of multiple earthquake events: find age spacing, and make sure both events have same spacing
        def jointprobability(self, other):
            diff_age = np.mean(np.diff(self.age))
            if diff_age != np.mean(np.diff(other.age)):
                raise ValueError("Sampling interval differs")
            #Calculate misallignment between both age groups, then use this to shift the two distributions so they line up
            misalign = (other.age[0] % diff_age) - (self.age[0] % diff_age)
            self_age = self.age.copy()
            other_age = other.age.copy()
            self_age = self_age + misalign
            #Find where distributions overlap
            lower = max(min(self_age), min(other_age))
            upper = min(max(self_age), max(other_age))

            if lower >= upper:
                return None
            #Create grid of shared ages
            new_age = np.arange(lower, upper + diff_age, diff_age)
            #Find starting and ending indexes for both event PDFs
            this_min = np.argmin(np.abs(self_age - lower))
            that_min = np.argmin(np.abs(other_age - lower))
            this_max = np.argmin(np.abs(self_age - upper))
            that_max = np.argmin(np.abs(other_age - upper))
            #Multiply PDFs element-wise
            new_pdf = self.PDF[this_min:this_max+1] * other.PDF[that_min:that_max+1]
            #Normalize (set sum of new PDF to 1)
            overlap = np.sum(new_pdf)
            new_pdf = new_pdf / overlap
            new_cdf = np.cumsum(new_pdf)

            return Earthquake(self.name, new_age, new_pdf, new_cdf)


    #Define function load_oxcal: Reads in events from each Oxcal file, pulls out and saves ages, pdf, and cdf data (outputs 'Earthquake' object)
    def load_oxcal(filename, name_selector='Event', delim=None):
        if delim:
            df = pd.read_csv(filename, sep=delim)
        else:
            df = pd.read_csv(filename)
    
        # Keep only posterior type
        df = df[df["type"] == "posterior"]
        df = df[df["name"].str.contains(name_selector)]

        result = []

        for eventname, group in df.groupby("name"):
            ages = group["value"].values
            pdf = group["probability"].values
            cdf = np.cumsum(pdf)
            sumcdf = cdf[-1]

            result.append(
                Earthquake(
                    name=eventname,
                    age=ages,
                    PDF=pdf / sumcdf,
                    CDF=cdf / sumcdf
                )
            )
        return result


        for eventname in events["name"].unique():
            event = events[events["name"] == eventname]


            ages = event["value"].values
            pdf = event["probability"].values
            cdf = np.cumsum(pdf)

            sumcdf = cdf[-1]
            result.append(
                Earthquake(
                    name=eventname,
                    age=ages,
                    PDF=pdf / sumcdf,
                    CDF=cdf / sumcdf
                )
            )
        return result


    #Define function get_marginal_pdfs: Builds dataframe with age, PDF, and name information as columns in dataframe with events as rows, 
    #also only keeps nonzero rows, and gives each event its own number
    def get_marginal_pdfs(oxcal):
        rows = []
        for i, eq in enumerate(oxcal):
            df = pd.DataFrame({
                "age": eq.age,
                "PDF": eq.PDF,
                "name": eq.name
            })
            df = df[df["age"] > 0]
            df["site_order"] = i
            rows.append(df)
        return rows

    #Define function as_binary_matrix: loops over dataframe to convert each row into binary
    def as_binary_matrix(n, width):
        return np.array([
            list(np.binary_repr(i, width=width))
            for i in range(1, n + 1)
        ], dtype=int)
##
#Original location for 'define sites dataframe'
##
    #Create marginal PDF dataframes for each paleoseismic site using load_oxcal function
    marginal_data = []

    for i, row in sites.iterrows():
        if pd.isna(row["file"]):
            df = pd.DataFrame({"age": [np.nan], "PDF": [np.nan], "name": [np.nan], "CDF": [np.nan]})
        else:
            oxcal = load_oxcal(row["file"], row["event_code"])
            df = pd.concat(get_marginal_pdfs(oxcal), ignore_index=True)

        df["site"] = row["site"]
        marginal_data.append(df)

    sites["marginal_pdf"] = marginal_data

    #Convert marginal_pdf to list, dropping column and row names
    marginal = pd.concat(sites["marginal_pdf"].tolist(), ignore_index=True)
    marginal = marginal.dropna(subset=["name"])

    #Converts data type to float, exchanges 'age' and 'PDF' index with 'name' and 'event'
    marginal["year"] = marginal["age"].astype(float)
    marginal["prob"] = marginal["PDF"].astype(float)
    marginal = marginal.drop(columns=["age", "PDF"])
    marginal = marginal.rename(columns={"name": "event"}) 

    #Format each trench event to be labeled as site.event_# (i.e. Bidart.Event_2)
    marginal["event"] = marginal["event"].str.replace(r"\s+", "_", regex=True)
    marginal["event_name"] = marginal["site"] + "_" + marginal["event"]

    print("Marginal dataframe constructed...")


##Added sort=False because if groupby sorts alphabetically, indexing for jointprobability will be screwed up
    #Calculate mean year for each event (weighted by probability)
    events = (
        marginal.groupby(["site", "event"], sort=False)
        .apply(lambda x: np.average(x["year"], weights=x["prob"]))
        .reset_index(name="year")
    )
    n_events = len(events)

    ##Moved drop historical events code here and edited to work on events
    #Removes historical events
    historical = marginal[marginal["prob"] == 1]["event_name"].unique()
    #Specific to San Andreas Fault: Remove EL1 and Event_X (known to be 1857 and 1812 historical events, respectively)
    historical = np.append(historical, 'Elizabeth_Event_EL1')
    historical = np.append(historical, 'Pallett_Event_X')

    events_drop_indices = []
    for event in historical:
        events_drop_indices.append(events[(events['site'] == event.split('_')[0]) & (events['event'] == event.split('_', 1)[-1])].index[0])
    #Specific to San Andreas fault: Drop index = 5, which is the historical Annette groundshaking event
    events_drop_indices.append(5)

    ##Drop all events in events_drop_indices (append low quality event indices above)
    events.drop(events.index[events_drop_indices], inplace = True)

    #Reset indices after dropping historical events
    events.reset_index(drop=True, inplace=True)

    #To be used later: event count after deletion of historical events
    n_events = len(events)

##

    #Build candidates matrix, using similar formatting from above to label event_names, then initiliazing candidates as an identity matrix
    event_names = (events["site"] + "_" + events["event"]).values
    candidates = np.zeros((n_events, n_events), dtype=int)
    np.fill_diagonal(candidates, 1)

    #Finds where two different event's PDFs overlap: goes through matrix of allevents x allevents, 
    #marking '1' in the position of the matrix where two different events overlap in time (similar idea to adjacency matrix)
    for i in range(n_events):
        for j in range(n_events):
    #Skips diagonals and where an event overlaps with itself
            if i == j:
                continue

            if events.loc[i, "site"] == events.loc[j, "site"]:
                continue

            yr1 = marginal[
                (marginal["site"] == events.loc[i, "site"]) &
                (marginal["event"] == events.loc[i, "event"]) &
                (marginal["prob"] > threshold)
            ]["year"].values

            yr2 = marginal[
                (marginal["site"] == events.loc[j, "site"]) &
                (marginal["event"] == events.loc[j, "event"]) &
                (marginal["prob"] > threshold)
            ]["year"].values

            if (yr1.min() < yr2.max()) and (yr2.min() < yr1.max()):
                candidates[i, j] = 1
                candidates[j, i] = 1
    
    print("Candidates built...")

##Create the equivalent of site_binary_matrix but for the candidates matrix
    candidates_list = [[] for site in range(len(sites['site']) - 2)]
    for idx, trench in enumerate(sites['site'][1:-1]):
        indices = []
        for j in range(events.shape[0]):
            if events['site'][j] == trench:
                indices.append(j)
        for i in range(events.shape[0]):
           #Note: events from the same site can appear multiple times in a row if multiple events from another trench overlap them
        #Therefore, to convert to binary, make condition that if sum of indexed range is nonzero, position in candidates_binary is 1, else it is 0
            if (candidates[i][np.min(indices) : np.max(indices) + 1]).sum() > 0:
                candidates_list[idx].append(1)
            else:
                candidates_list[idx].append(0)

##Turn list of lists into matrix and transpose to get in same format as site_binary_matrix
    candidates_matrix = np.array(candidates_list)
    candidates_binary = candidates_matrix.T

#Make arrays of arrays (matrix) that just hold the indices of each trench site, in same order (to be used to figure out where to split each row of caniddates)
    site_indices = [[] for site in range(len(sites) - 2)]
    for idx, site in enumerate(sites['site'][1:-1]):
        site_indices[idx].append(np.where(events['site'] == site)[0])

##Code to find invalid gaps: restructure and use to filter candidates BEFORE creating all possible combinations

#Make new matrix to store the new valid rows created from candidates
#Make new matrix to store the new valid rows created from candidates
    new_candidates = []

    for i in range(len(candidates_binary)):
    #Ensure there are at least two 1s to have something "in-between"
        if np.count_nonzero(candidates_binary[i]) <= 1:
            new_candidates.append(candidates[i])
        else:
        #Find the boundary indexes of the first and last '1'
            all_ones = (np.where(candidates_binary[i] == 1))[0]
            first_one = all_ones[0]
            last_one = all_ones[-1]
        #Check if 0 exists inside this sliced sub-array
            middle = candidates_binary[i][first_one + 1 : last_one]
            if np.any(middle == 0):
        #Pasted from def valid_overlap: calculate the age range of the earthquake represented by this row
                cols = np.where(candidates[i] == 1)[0]
                selected = event_names[cols]
                dfs = [
                    marginal[
                        (marginal["event_name"] == e) &
                                (marginal["prob"] > threshold)
                            ]
                        for e in selected
                        ]
                mins = [df["year"].min() for df in dfs]
                maxs = [df["year"].max() for df in dfs]

                event_min = max(mins)
                event_max = min(maxs)

        #Find the youngest date of the age limits of the two records on either side by finding the youngest of the two oldest dates from each of the two record's PDFs
        ##Added + 1 to both first_one and last_one because marginal['site'] has length 10 and candidates_binary has length 8
                youngest_adjacent_record_base = max(min(marginal.loc[marginal['site'] == sites['site'][first_one + 1], 'year']), min(marginal.loc[marginal['site'] == sites['site'][last_one + 1], 'year']))
        #Find indexes of 0's in middle (to be used in next step to test if the gap of 0's is from only prerecord sites)
                ##IMPORTANT EDIT: Remove back of zeros_idx code
                zeros_idx = np.where(middle == 0)[0]# + (first_one)
##IMPORTANT EDIT: First one + 1 is needed because we're trying to find the actual index of the zero based on its index in the middle
        #(+1) and the index of the first element of the gap (first_one)
                zeros_idx = [x + (first_one + 1) for x in zeros_idx]
        #Make array of the oldest age of each site that contains zero in this middle gap
                min_ages = []
                for index in zeros_idx:
                    min_ages.append((marginal.loc[marginal['site'] == sites['site'][index + 1], 'year']).min())

                youngest_adjacent_record_base = []
                for index in zeros_idx:
                    ones_index = np.searchsorted(all_ones, index)
                    idx_below = all_ones[ones_index - 1]
                    idx_above = all_ones[ones_index]
                    adjacent_record_base = max(min(marginal.loc[marginal['site'] == sites['site'][idx_below + 1], 'year']), (min(marginal.loc[marginal['site'] == sites['site'][idx_above + 1], 'year']))) 
                    youngest_adjacent_record_base.append(adjacent_record_base)
        #Restructured from earlier version of code: if there is an invalid gap in this row, split up row at invalid gaps into seperate rows
        #Condition structured as 'if there is a zero in this middle gap AND the shortest of the two records on either side is still 
        #longer than the length of any of the records that are zero in this gap AND the age range of the earthquake represented by this row is older 
        #than the oldest part of any of the records that are 0 in this gap' then this is not valid for a single event and must be split up

        ##Plan: store the first and last indices of each site from events (same as original candidates), then use above condition to find
        #and store the indices of each gap, which can be matched back to index of trench, which can be matched back to the index to split the
        #row of candidates by, then add 0's back to rows, then replace original row with these split rows
                
                invalid_binary_gaps_indices = []
                for n in range(len(zeros_idx)):
                    if min_ages[n] < youngest_adjacent_record_base[n] or event_min > min_ages[n]:
                        invalid_binary_gaps_indices.append(zeros_idx[n])
        ##Added condition so that if there is a gap but it's a valid gap (leaving invalid_binary_gaps_indices empty), 
        #append the original candidates row and skip below
                if not invalid_binary_gaps_indices:
                    new_candidates.append(candidates[i])
                    continue
                split_indices = []
                for index in invalid_binary_gaps_indices:
                    split_indices.append(site_indices[index][0][0])
                for index in split_indices:
                    row = candidates[i]
                    row1 = row.copy()
                    row1[index:] = 0
                    row2 = row.copy()
                    row2[:index] = 0
                    new_candidates.append(row1)
                    new_candidates.append(row2)

###
            else:
                new_candidates.append(candidates[i])
###
    
    #Generates all possible combinations of events: finds the indices where each row contains 1s (idx), skips rows with no 1s, 
    all_combinations = []

    for i in range(n_events):
        idx = np.where(candidates[i] == 1)[0]
        k = len(idx)

        if k == 0:
            continue
        #Creates all possible combinations of 1s and 0s (except for the all zero combination, hence '-1')
        combos = as_binary_matrix(2**k - 1, k)
        #Initialize matrix: matrix of all 0, with columns being the events and rows the possible combinations
        mat = np.zeros((combos.shape[0], n_events), dtype=int)
        #Creates matrix holding all of the combinations that have 1s in the same position (index) specified by 'idx'
        mat[:, idx] = combos

        all_combinations.append(mat)
    #Combines all into one large matrix
    all_combinations = np.vstack(all_combinations)

    #Removes historical events
    historical = marginal[marginal["prob"] == 1]["event_name"].unique()
    #Specific to San Andreas Fault: Remove EL1 and Event_X (known to be 1857 and 1812 historical events, respectively)
    historical = np.append(historical, 'Elizabeth_Lake_Event_EL1')
    historical = np.append(historical, 'Pallett_Creek_Event_X')

    cols_keep = [i for i, e in enumerate(event_names) if e not in historical]

    event_keep_names = []
    for i in cols_keep:
        event_keep_names.append(event_names[i])
    #Remove the rows that turned into all zero rows after the removal of historical events
    for i in range(len(all_combinations)):
        if sum(all_combinations[i]) == 0:
            all_combinations = np.delete(all_combinations, i, axis = 0)

    
    #Removes candidate events that include more than one event from a site
    sites_sorted = sites.sort_values("loc")
    site_names = sites_sorted["site"].values
    #Will store how many events from each site are in combination
    site_count = np.zeros((all_combinations.shape[0], len(site_names)))

    for i, site in enumerate(site_names):
        cols = [c for c in event_keep_names if c.startswith(site + "_")]
        idx = [np.where(event_names == c)[0][0] for c in cols]
        site_count[:, i] = all_combinations[:, idx].sum(axis=1)
    #Only keep events with <= 1 event per site
    mask = (site_count <= 1).all(axis=1)
    all_combinations = all_combinations[mask]
    site_count = site_count[mask]

    print("All combinations finished...")

    
## New new version to drop candidate events with gaps UNLESS that gap is from a site in the time before its record started

    drop_indices = []

    for i in range(len(all_combinations)):
    #Ensure there are at least two 1s to have something "in-between"
        if np.count_nonzero(site_count[i]) > 1:
        #Find the boundary indexes of the first and last '1'
            all_ones = (np.where(site_count[i] == 1))[0]
            first_one = all_ones[0]
            last_one = all_ones[-1]
        #Check if 0 exists inside this sliced sub-array
            middle = site_count[i][first_one + 1 : last_one]
            if np.any(middle == 0):
        ##Pasted from def valid_overlap: calculate the age range of the earthquake represented by this row
                cols = np.where(all_combinations[i] == 1)[0]
                selected = event_names[cols]
                dfs = [
                    marginal[
                        (marginal["event_name"] == e) &
                                (marginal["prob"] > threshold)
                            ]
                        for e in selected
                        ]
                mins = [df["year"].min() for df in dfs]
                maxs = [df["year"].max() for df in dfs]

                event_min = max(mins)
                event_max = min(maxs)
   
        #Find the youngest date of the age limits of the two records on either side by finding the youngest of the two oldest dates from each of the two record's PDFs
                youngest_adjacent_record_base = max(min(marginal.loc[marginal['site'] == sites['site'][first_one], 'year']), min(marginal.loc[marginal['site'] == sites['site'][last_one], 'year']))
        #Find indexes of 0's in middle (to be used in next step to test if the gap of 0's is from only prerecord sites)
                zeros_idx = np.where(middle == 0)[0] + (first_one)
        #Make array of the oldest age of each site that contains zero in this middle gap
                min_ages = []
                for index in zeros_idx[0]:
                    min_ages.append((marginal.loc[marginal['site'] == sites['site'][index], 'year']).min())
        #Condition structured as 'if there is a zero in this middle gap AND the shortest of the two records on either side is still 
        #longer than the length of any of the records that are zero in this gap AND the age range of the earthquake represented by this row is older 
        #than the oldest part of any of the records that are 0 in this gap' then this event is not added to drop_indices
                if np.any(np.array(min_ages, dtype = float) < int(youngest_adjacent_record_base)) or np.any(event_min > np.array(min_ages, dtype = float)):
                    drop_indices.append(i)
                    

    valid_combinations = np.delete(all_combinations, drop_indices, axis = 0)

##

    #Remove duplicate rows
    valid_combinations = np.unique(valid_combinations, axis=0)

    print("Valid combinations finished...")
    

    #Define function valid_overlap: find candidates where different trench events overlap in time
    def valid_overlap(row, threshold=0):
        cols = np.where(row == 1)[0]
        selected = event_names[cols]
    
        dfs = [
            marginal[
                (marginal["event_name"] == e) &
                    (marginal["prob"] > threshold)
            ]
            for e in selected
        ]

        if any(df.empty for df in dfs):
            return False

        mins = [df["year"].min() for df in dfs]
        maxs = [df["year"].max() for df in dfs]

        return max(mins) < min(maxs)


    #Use to find all candidate events whose constitive earthquake events have statistically significant overlap


## Original location for historical code through valid_combinations code
    mask = np.array([valid_overlap(r, threshold) for r in valid_combinations])
    valid_combinations = valid_combinations[mask]

    valid_combinations = valid_combinations[:, cols_keep]
#Remove rows of zeros from valid combinations
    valid_combinations = [row for row in valid_combinations if sum(row)>0]

    
    #Put valid_combinations into DataFrame to create 'candidates'
    candidates_df = pd.DataFrame(valid_combinations, columns=event_names[cols_keep])
    candidates_df["event_id"] = np.arange(1, len(candidates_df) + 1)

    print("Candidates dataframe finished, creating earthquake objects for each candidate...")
    


if Enumeration_Option == 2:
    #Create earthquake class
    class Earthquake:

        #Constructor: stores earthquake name, age, PDF, and CDF for each earthquake event (i.e. Bidart.Event_2)
        def __init__(self, name, age, PDF, CDF):
            self.name = name
            self.age = np.array(age)
            self.PDF = np.array(PDF)
            self.CDF = np.array(CDF)
        #Returns single sampled age based on PDF: generate random number between 0 and 1, then sample the age from the CDF that corresponds to number
        def event(self):
            r = np.random.rand()
            return self.age[np.argmin(np.abs(self.CDF - r))]
        #Returns n sampled ages: calls event() n times, then stores results in an array
        def events(self, n):
            return np.array([self.event() for _ in range(n)])
        #Resamples earthquake age to yearly resolution: calculates age range (L), then creates list of years inside age range and interpolates event PDF and CDF to those years
        def resample(self):
            L = int(self.age[-1] - self.age[0])
            new_age = np.linspace(self.age[0], self.age[-1], L + 1)

            f_age = interp1d(self.age, self.age, fill_value="extrapolate")
            f_pdf = interp1d(self.age, self.PDF, fill_value="extrapolate")
            f_cdf = interp1d(self.age, self.CDF, fill_value="extrapolate")

            self.age = f_age(new_age)
            self.PDF = f_pdf(new_age)
            self.PDF = self.PDF / np.sum(self.PDF)
            self.CDF = f_cdf(new_age)
        #Calculates joint probability of multiple earthquake events: find age spacing, and make sure both events have same spacing
        def jointprobability(self, other):
            diff_age = np.mean(np.diff(self.age))
            if diff_age != np.mean(np.diff(other.age)):
                raise ValueError("Sampling interval differs")
            #Calculate misallignment between both age groups, then use this to shift the two distributions so they line up
            misalign = (other.age[0] % diff_age) - (self.age[0] % diff_age)
            self_age = self.age.copy()
            other_age = other.age.copy()
            self_age = self_age + misalign
            #Find where distributions overlap
            lower = max(min(self_age), min(other_age))
            upper = min(max(self_age), max(other_age))

            if lower >= upper:
                return None
            #Create grid of shared ages
            new_age = np.arange(lower, upper + diff_age, diff_age)
            #Find starting and ending indexes for both event PDFs
            this_min = np.argmin(np.abs(self_age - lower))
            that_min = np.argmin(np.abs(other_age - lower))
            this_max = np.argmin(np.abs(self_age - upper))
            that_max = np.argmin(np.abs(other_age - upper))
            #Multiply PDFs element-wise
            new_pdf = self.PDF[this_min:this_max+1] * other.PDF[that_min:that_max+1]
            #Normalize (set sum of new PDF to 1)
            overlap = np.sum(new_pdf)
            new_pdf = new_pdf / overlap
            new_cdf = np.cumsum(new_pdf)

            return Earthquake(self.name, new_age, new_pdf, new_cdf)


    #Define function load_oxcal: Reads in events from each Oxcal file, pulls out and saves ages, pdf, and cdf data (outputs 'Earthquake' object)
    def load_oxcal(filename, name_selector='Event', delim=None):
        if delim:
            df = pd.read_csv(filename, sep=delim)
        else:
            df = pd.read_csv(filename)
    
        # Keep only posterior type
        df = df[df["type"] == "posterior"]
        df = df[df["name"].str.contains(name_selector)]

        result = []

        for eventname, group in df.groupby("name"):
            ages = group["value"].values
            pdf = group["probability"].values
            cdf = np.cumsum(pdf)
            sumcdf = cdf[-1]

            result.append(
                Earthquake(
                    name=eventname,
                    age=ages,
                    PDF=pdf / sumcdf,
                    CDF=cdf / sumcdf
                )
            )
        return result


        for eventname in events["name"].unique():
            event = events[events["name"] == eventname]


            ages = event["value"].values
            pdf = event["probability"].values
            cdf = np.cumsum(pdf)

            sumcdf = cdf[-1]
            result.append(
                Earthquake(
                    name=eventname,
                    age=ages,
                    PDF=pdf / sumcdf,
                    CDF=cdf / sumcdf
                )
            )
        return result


    #Define function get_marginal_pdfs: Builds dataframe with age, PDF, and name information as columns in dataframe with events as rows, 
    #also only keeps nonzero rows, and gives each event its own number
    def get_marginal_pdfs(oxcal):
        rows = []
        for i, eq in enumerate(oxcal):
            df = pd.DataFrame({
                "age": eq.age,
                "PDF": eq.PDF,
                "name": eq.name,
                ##
                "CDF": eq.CDF
                ##
            })
            df = df[df["age"] > 0]
            df["site_order"] = i
            rows.append(df)
        return rows

    #Define function as_binary_matrix: loops over dataframe to convert each row into binary
    def as_binary_matrix(n, width):
        return np.array([
            list(np.binary_repr(i, width=width))
            for i in range(1, n + 1)
        ], dtype=int)

##
#Original location for 'define sites dataframe'
##

    #Create marginal PDF dataframes for each paleoseismic site using load_oxcal function
    marginal_data = []

    for i, row in sites.iterrows():
        if pd.isna(row["file"]):
            df = pd.DataFrame({"age": [np.nan], "PDF": [np.nan], "name": [np.nan]})
        else:
            oxcal = load_oxcal(row["file"], row["event_code"])
            df = pd.concat(get_marginal_pdfs(oxcal), ignore_index=True)

        df["site"] = row["site"]
        marginal_data.append(df)

    sites["marginal_pdf"] = marginal_data

    #Convert marginal_pdf to list, dropping column and row names
    marginal = pd.concat(sites["marginal_pdf"].tolist(), ignore_index=True)
    marginal = marginal.dropna(subset=["name"])

    #Converts data type to float, exchanges 'age' and 'PDF' index with 'name' and 'event'
    marginal["year"] = marginal["age"].astype(float)
    marginal["prob"] = marginal["PDF"].astype(float)
    marginal = marginal.drop(columns=["age", "PDF"])
    marginal = marginal.rename(columns={"name": "event"}) 

    #Format each trench event to be labeled as site.event_# (i.e. Bidart.Event_2)
    marginal["event"] = marginal["event"].str.replace(r"\s+", "_", regex=True)
    marginal["event_name"] = marginal["site"] + "_" + marginal["event"]

    print("Marginal dataframe constructed...")

#Added sort=False because if groupby sorts alphabetically, indexing for jointprobability will be screwed up
    #Calculate mean year for each event (weighted by probability)
    events = (
        marginal.groupby(["site", "event"], sort=False)
        .apply(lambda x: np.average(x["year"], weights=x["prob"]))
        .reset_index(name="year")
    )

##Moved drop historical events code here and edited to work on events
    #Removes historical events
    historical = marginal[marginal["prob"] == 1]["event_name"].unique()
    #Specific to San Andreas Fault: Remove EL1 and Event_X (known to be 1857 and 1812 historical events, respectively)
    historical = np.append(historical, 'Elizabeth_Event_EL1')
    historical = np.append(historical, 'Pallett_Event_X')

    events_drop_indices = []
    for event in historical:
        events_drop_indices.append(events[(events['site'] == event.split('_')[0]) & (events['event'] == event.split('_', 1)[-1])].index[0])
    #ONLY ACTIVATE BELOW LINE WHEN ANNETTE IS INCLUDED IN MODEL: Specific to San Andreas fault: Drop index = 5, which is the historical Annette groundshaking event
    events_drop_indices.append(5)

    ##Drop all events in events_drop_indices (append low quality event indices above)
    events.drop(events.index[events_drop_indices], inplace = True)

    #Reset indices after dropping historical events
    events.reset_index(drop=True, inplace=True)

    #To be used later: event count after deletion of historical events
    n_events = len(events)

##

    #Build candidates matrix, using similar formatting from above to label event_names, then initiliazing candidates as an identity matrix
    event_names = (events["site"] + "_" + events["event"]).values
    candidates = np.zeros((n_events, n_events), dtype=int)
    np.fill_diagonal(candidates, 1)

    #Finds where two different event's PDFs overlap: goes through matrix of allevents x allevents, 
    #marking '1' in the position of the matrix where two different events overlap in time (similar idea to adjacency matrix)
    for i in range(n_events):
        for j in range(n_events):
            if i == j:
                continue

            if events.loc[i, "site"] == events.loc[j, "site"]:
                continue

            yr1 = marginal[
                (marginal["site"] == events.loc[i, "site"]) &
                (marginal["event"] == events.loc[i, "event"]) &
                (marginal["CDF"] > threshold) &
                (marginal["CDF"] < 1 - threshold)
            ]["year"].values

            yr2 = marginal[
                (marginal["site"] == events.loc[j, "site"]) &
                (marginal["event"] == events.loc[j, "event"]) &
                (marginal["CDF"] > threshold) &
                (marginal["CDF"] < 1 - threshold)
            ]["year"].values

            if (yr1.min() < yr2.max()) and (yr2.min() < yr1.max()):
                candidates[i, j] = 1
                candidates[j, i] = 1

    print("Candidates built...")

##Create the equivalent of site_binary_matrix but for the candidates matrix
    candidates_list = [[] for site in range(len(sites['site']) - 2)]
    for idx, trench in enumerate(sites['site'][1:-1]):
        indices = []
        for j in range(events.shape[0]):
            if events['site'][j] == trench:
                indices.append(j)
        for i in range(events.shape[0]):
           #Note: events from the same site can appear multiple times in a row if multiple events from another trench overlap them
        #Therefore, to convert to binary, make condition that if sum of indexed range is nonzero, position in candidates_binary is 1, else it is 0
            if (candidates[i][np.min(indices) : np.max(indices) + 1]).sum() > 0:
                candidates_list[idx].append(1)
            else:
                candidates_list[idx].append(0)

##Turn list of lists into matrix and transpose to get in same format as site_binary_matrix
    candidates_matrix = np.array(candidates_list)
    candidates_binary = candidates_matrix.T

#Make arrays of arrays (matrix) that just hold the indices of each trench site, in same order (to be used to figure out where to split each row of caniddates)
    site_indices = [[] for site in range(len(sites) - 2)]
    for idx, site in enumerate(sites['site'][1:-1]):
        site_indices[idx].append(np.where(events['site'] == site)[0])

##Code to find invalid gaps: restructure and use to filter candidates BEFORE creating all possible combinations

#Make new matrix to store the new valid rows created from candidates
    new_candidates = []

    for i in range(len(candidates_binary)):
    #Ensure there are at least two 1s to have something "in-between"
        if np.count_nonzero(candidates_binary[i]) <= 1:
            new_candidates.append(candidates[i])
        else:
        #Find the boundary indexes of the first and last '1'
            all_ones = (np.where(candidates_binary[i] == 1))[0]
            first_one = all_ones[0]
            last_one = all_ones[-1]
        #Check if 0 exists inside this sliced sub-array
            middle = candidates_binary[i][first_one + 1 : last_one]
            if np.any(middle == 0):
        #Pasted from def valid_overlap: calculate the age range of the earthquake represented by this row
                cols = np.where(candidates[i] == 1)[0]
                selected = event_names[cols]
                dfs = [
                    marginal[
                        (marginal["event_name"] == e) &
                        (marginal["CDF"] > threshold) &
                        (marginal["CDF"] < 1 - threshold)
                            ]
                        for e in selected
                        ]
                mins = [df["year"].min() for df in dfs]
                maxs = [df["year"].max() for df in dfs]

                event_min = max(mins)
                event_max = min(maxs)

        #Find the youngest date of the age limits of the two records on either side by finding the youngest of the two oldest dates from each of the two record's PDFs
        ##Added + 1 to both first_one and last_one because marginal['site'] has length 10 and candidates_binary has length 8
                youngest_adjacent_record_base = max(min(marginal.loc[marginal['site'] == sites['site'][first_one + 1], 'year']), min(marginal.loc[marginal['site'] == sites['site'][last_one + 1], 'year']))
        #Find indexes of 0's in middle (to be used in next step to test if the gap of 0's is from only prerecord sites)
                ##IMPORTANT EDIT: Remove back of zeros_idx code
                zeros_idx = np.where(middle == 0)[0]# + (first_one)
##IMPORTANT EDIT: First one + 1 is needed because we're trying to find the actual index of the zero based on its index in the middle
        #(+1) and the index of the first element of the gap (first_one)
                zeros_idx = [x + (first_one + 1) for x in zeros_idx]
        #Make array of the oldest age of each site that contains zero in this middle gap
                min_ages = []
                for index in zeros_idx:
                    min_ages.append((marginal.loc[marginal['site'] == sites['site'][index + 1], 'year']).min())

                youngest_adjacent_record_base = []
                for index in zeros_idx:
                    ones_index = np.searchsorted(all_ones, index)
                    idx_below = all_ones[ones_index - 1]
                    idx_above = all_ones[ones_index]
                    adjacent_record_base = max(min(marginal.loc[marginal['site'] == sites['site'][idx_below + 1], 'year']), (min(marginal.loc[marginal['site'] == sites['site'][idx_above + 1], 'year']))) 
                    youngest_adjacent_record_base.append(adjacent_record_base)
        #Restructured from earlier version of code: if there is an invalid gap in this row, split up row at invalid gaps into seperate rows
        #Condition structured as 'if there is a zero in this middle gap AND the shortest of the two records on either side is still 
        #longer than the length of any of the records that are zero in this gap AND the age range of the earthquake represented by this row is older 
        #than the oldest part of any of the records that are 0 in this gap' then this is not valid for a single event and must be split up

        ##Plan: store the first and last indices of each site from events (same as original candidates), then use above condition to find
        #and store the indices of each gap, which can be matched back to index of trench, which can be matched back to the index to split the
        #row of candidates by, then add 0's back to rows, then replace original row with these split rows
                
                invalid_binary_gaps_indices = []
                for n in range(len(zeros_idx)):
                    if min_ages[n] < youngest_adjacent_record_base[n] or event_min > min_ages[n]:
                        invalid_binary_gaps_indices.append(zeros_idx[n])
        ##Added condition so that if there is a gap but it's a valid gap (leaving invalid_binary_gaps_indices empty), 
        #append the original candidates row and skip below
                if not invalid_binary_gaps_indices:
                    new_candidates.append(candidates[i])
                    continue
                split_indices = []
                for index in invalid_binary_gaps_indices:
                    split_indices.append(site_indices[index][0][0])
                for index in split_indices:
                    row = candidates[i]
                    row1 = row.copy()
                    row1[index:] = 0
                    row2 = row.copy()
                    row2[:index] = 0
                    new_candidates.append(row1)
                    new_candidates.append(row2)

###
            else:
                new_candidates.append(candidates[i])
###

###New addition: create candidates binary for new candidates, to be used to decide which method of candidates generation will be used on the row
##Create the equivalent of site_binary_matrix but for the candidates matrix
    new_candidates_list = [[] for site in range(len(sites['site']) - 2)]
    for idx, trench in enumerate(sites['site'][1:-1]):
        indices = []
        for j in range(events.shape[0]):
            if events['site'][j] == trench:
                indices.append(j)
        for i in range(len(new_candidates)):
           #Note: events from the same site can appear multiple times in a row if multiple events from another trench overlap them
        #Therefore, to convert to binary, make condition that if sum of indexed range is nonzero, position in candidates_binary is 1, else it is 0
            if (new_candidates[i][np.min(indices) : np.max(indices) + 1]).sum() > 0:
                new_candidates_list[idx].append(1)
            else:
                new_candidates_list[idx].append(0)

##Turn list of lists into matrix and transpose to get in same format as site_binary_matrix
    new_candidates_matrix = np.array(new_candidates_list)
    new_candidates_binary = new_candidates_matrix.T
###

    #Generates all possible combinations of events: finds the indices where each row contains 1s (idx), skips rows with no 1s, 
    all_combinations = []

    for i in range(len(new_candidates)):
        ###New addition: print the current iteration of the loop to track where loop is getting stuck
        print("Current row of new_candidates:", i)
        idx = np.where(new_candidates[i] == 1)[0]
        k = len(idx)

        if k == 0:
            continue
###New addition: add condition so that if there are enough ones in a row to overflow the computer (~22), use my alternate code to generate all combinations
#with <= len(trench_events) (note: this alternate code is not run for all rows because it will also overflow computer)
#Must use if, elif, and else for these conditions
        elif sum(new_candidates_binary[i]) == len(sites['site']) - 2:
            print("Working on above-capacity row...")

#Make an array for each trench site

#Sort idx into an array of arrays where each array holds the indices that belong to that trench
            indices_sets = [[] for site in range(len(sites['site']) - 2)]
            for one_index in idx:
                for i in range(len(site_indices)): #Replace testarray with site_indices
                    for j in range(len(site_indices[i][0])):
                        if site_indices[i][0][j] == one_index: #Will need to use site_indices[i][0][j]
                            indices_sets[i].append(one_index)

#Second task: choose all combinations of one index from each set (see Google search)
            all_index_combinations = list(product(*indices_sets))

#Third task: sub all combinations of 1 and 0 into a row of zero for each combination from that set of index combinations

            for row in all_index_combinations:

        # Generate all 0/1 combinations for those indices
                        for combo in itertools.product(
                            [0, 1], repeat=len(row)
                        ):
    #Changed from variant = list(testrow) because if it is kept as a copy then there are rows with the original number of 1's
                            variant = np.zeros(len(events))

                # Change only the selected indices
                            for index, bit in zip(row, combo):
                                variant[index] = bit

                            all_combinations.append(variant)
###
        else:
        #Creates all possible combinations of 1s and 0s (except for the all zero combination, hence '-1')
            combos = as_binary_matrix(2**k - 1, k)
        #Initialize matrix: matrix of all 0, with columns being the events and rows the possible combinations
            mat = np.zeros((combos.shape[0], n_events), dtype=int)
        #Creates matrix holding all of the combinations that have 1s in the same position (index) specified by 'idx'
            mat[:, idx] = combos

            all_combinations.append(mat)
    #Combines all into one large matrix
    all_combinations = np.vstack(all_combinations)
    print("Finished first draft of all_combinations...")
    #Removes historical events
    historical = marginal[marginal["prob"] == 1]["event_name"].unique()
    #Specific to San Andreas Fault: Remove EL1 and Event_X (known to be 1857 and 1812 historical events, respectively)
    #Remove historical groundshaking event from Annette site
    historical = np.append(historical, 'Elizabeth_Lake_Event_EL1')
    historical = np.append(historical, 'Pallett_Creek_Event_X')
    historical = np.append(historical, 'Groundshaking_Event_')

    cols_keep = [i for i, e in enumerate(event_names) if e not in historical]
    print("Historical events filtered...")

    event_keep_names = []
    for i in cols_keep:
        event_keep_names.append(event_names[i])
    print("event_keep_names created...")
    #Remove the rows that turned into all zero rows after the removal of historical events
    all_combinations = [row for row in all_combinations if sum(row) != 0]
    print("Removed zero rows")

    ###New addition: convert to numpy matrix for use below
    all_combinations = np.array(all_combinations)
    
    #Removes candidate events that include more than one event from a site
    sites_sorted = sites.sort_values("loc")
    site_names = sites_sorted["site"].values
    #Will store how many events from each site are in combination
    site_count = np.zeros((len(all_combinations), len(site_names)))

    for i, site in enumerate(site_names):
        cols = [c for c in event_keep_names if c.startswith(site + "_")]
        idx = [np.where(event_names == c)[0][0] for c in cols]
        site_count[:, i] = all_combinations[:, idx].sum(axis=1)
    print("Site_count created")
    #Only keep events with <= 1 event per site
    mask = (site_count <= 1).all(axis=1)
    all_combinations = all_combinations[mask]
    site_count = site_count[mask]

    print("All combinations finished...")

###TEST: Newest version of second gap filter and valid overlap filter

    event_lower_ages = np.zeros(len(events))
    event_upper_ages = np.zeros(len(events))
    trench_oldest_ages = []

    for i, event in enumerate(event_names):
        event_lower_ages[i] = min(marginal[(marginal["event_name"] == event) & (marginal["CDF"] > threshold) & (marginal["CDF"] < 1 - threshold)]['year'])
        event_upper_ages[i] = max(marginal[(marginal["event_name"] == event) & (marginal["CDF"] > threshold) & (marginal["CDF"] < 1 - threshold)]['year'])

    for file in sites['file'][1:-1]:
        df = pd.read_csv(file)
        trench_oldest_ages.append(min(df.loc[(df['op'] == 'boundary') | (df['op'] == 'Boundary') & (df['probability'] > prob_threshold), 'value']))

    #Convert trench_oldest_ages to numpy array so list indexing works in next step
    trench_oldest_ages = np.array(trench_oldest_ages)

    drop_indices = []

    for i in range(len(all_combinations)):
    #Ensure there are at least two 1s to have something "in-between"
        if np.count_nonzero(site_count[i]) > 1:
        #Find the boundary indexes of the first and last '1'
            all_ones = (np.where(site_count[i] == 1))[0]
            first_one = all_ones[0]
            last_one = all_ones[-1]
        #Check if 0 exists inside this sliced sub-array
            middle = site_count[i][first_one + 1 : last_one]
            if np.any(middle == 0):
        ##Pasted from def valid_overlap: calculate the age range of the earthquake represented by this row
                cols = np.where(all_combinations[i] == 1)[0]
                event_min = max(event_lower_ages[cols])
                event_max = min(event_upper_ages[cols])
   
#Important note: the indexing for marginal is shifted down by one in this part of the code because site_count has the same dimension as marginal        
        #Find indexes of 0's in middle (to be used in next step to test if the gap of 0's is from only prerecord sites)
##Important Note: Indexing for zeros_idx is different in this gap filter (changed to x + (first_one)) compared to first gap filter because this one
##is indexing off of site_count instead of candidates binary
                zeros_idx = np.where(middle == 0)[0]
                zeros_idx = [x + (first_one) for x in zeros_idx]
        #Make array of the oldest age of each site that contains zero in this middle gap
                min_ages = trench_oldest_ages[zeros_idx]
        #Find youngest of two records on sides of first and last zero: Will this break if there are two zero gaps 
####IMPORTANT EDIT: New version: find the indices of the 1's that bound each zero, and index marginal based on the indices of those 1's
#In other words, find the youngest adjacent record for each zero (to be used in next step)
                youngest_adjacent_record_base = []
                for index in zeros_idx:
                    ones_index = np.searchsorted(all_ones, index)
                    idx_below = all_ones[ones_index - 1]
                    idx_above = all_ones[ones_index]
                    adjacent_record_base = max(trench_oldest_ages[idx_below - 1], trench_oldest_ages[idx_above - 1]) 
                    youngest_adjacent_record_base.append(adjacent_record_base)            
        #Condition structured as 'if there is a zero in this middle gap AND the shortest of the two records on either side is still 
        #longer than the length of any of the records that are zero in this gap AND the age range of the earthquake represented by this row is older 
        #than the oldest part of any of the records that are 0 in this gap' then this event is not added to drop_indices
                for n in range(len(zeros_idx)):
                    if min_ages[n] < youngest_adjacent_record_base[n] or event_min > min_ages[n]: #If gap is invalid under event timing and trench record times, test if gap can be explained by a depositional hiatus
                        trench = trench_names[zeros_idx[n]]
                        test = Sedimentary_Hiatus_df.index[(Sedimentary_Hiatus_df['trench'] == trench) & (Sedimentary_Hiatus_df['upper age'] > event_max) & (Sedimentary_Hiatus_df['lower age'] < event_min)]
                        if len(test) == 0:
                            drop_indices.append[i]

###IMPORTANT EDIT: Added condition so that if there are multiple trench events from the same site in the same candidate event, add to drop_indices
        for j in range(len(site_indices)):
            if sum(all_combinations[i][site_indices[j][0][0]:site_indices[j][0][-1]]) > 1:
                drop_indices.append(i)
###
                    

    valid_combinations = np.delete(all_combinations, drop_indices, axis = 0)

##

    #Remove duplicate rows
    valid_combinations = np.unique(valid_combinations, axis=0)

    print("Valid combinations finished...")
    print("Dropped:", len(drop_indices))

###Convert to array of integers (binary) instead of floats:
    valid_combinations = valid_combinations.astype(int)
###New code to filter events with valid_overlap
    drop_indices = []

    for i in range(len(valid_combinations)):
        event_indices = np.where(valid_combinations[i] == 1)[0]
        event_oldest = np.max(event_lower_ages[event_indices])
        event_youngest = np.min(event_upper_ages[event_indices])
        if event_oldest >= event_youngest:
            drop_indices.append(i)

    valid_combinations = np.delete(
        valid_combinations,
        drop_indices,
        axis=0
    )

    print("Valid combinations finished...")

#Remove rows of zeros from valid combinations
    valid_combinations = [row for row in valid_combinations if sum(row) != 0]

    
    #Put valid_combinations into DataFrame to create 'candidates'
    candidates_df = pd.DataFrame(valid_combinations, columns=event_names[cols_keep])
    candidates_df["event_id"] = np.arange(1, len(candidates_df) + 1)

    print("Candidates dataframe finished, creating earthquake objects for each candidate...")

all_oxcal = []

for i, row in sites.iterrows():
    if not pd.isna(row["file"]):
        oxcal = load_oxcal(row["file"], row["event_code"])
        all_oxcal.extend(oxcal)

#Stores 55 events due to 6 historic events + duplicated (14) Wrightwood events
#Delete historical events from all_oxcal (using the fact that their ages will be whole-number years)
for n in reversed(range(len(all_oxcal))):
    all_whole = all(x % 1 == 0 for x in all_oxcal[n].age)
    if all_whole == True:
        del all_oxcal[n]

#Specific for San Andreas Fault since there are extra Wrightwood events in csv for some reason (28 instead of 14 with names like 'Event W510')
all_oxcal = [n for n in all_oxcal if len(n.name) < 10]

#Specific for San Andreas Fault: remove Pallett Event X and Elizabeth Lake Event EL1 from all_oxcal since it is now known as the historical 1812 earthquake
for i in range(len(all_oxcal)-2):
    if np.mean(all_oxcal[i].age) == 1748.0 and all_oxcal[i].name == 'Event X' or all_oxcal[i].name == 'Event EL1':
        all_oxcal.pop(i)

candidate_earthquakes = {}
for i in range(candidates_df.shape[0]):
    row = candidates_df.iloc[i,:-1].to_numpy()
    ones_indices_ce = np.nonzero(row)[0]
    joint = all_oxcal[ones_indices_ce[0]]
    if np.sum(row) == 1:
        key = f"individual{i}"
        candidate_earthquakes[key] = joint
    if np.sum(row) >= 2:
        for idx in ones_indices_ce[1:]:
            joint = joint.jointprobability(all_oxcal[idx])
            key = f"joint{i}"
            candidate_earthquakes[key] = joint



candidate_earthquake_ages = np.zeros(len(list(candidate_earthquakes)))
for i in range(len(list(candidate_earthquakes))):
    name = list(candidate_earthquakes)[i]
    candidate_earthquake_ages[i] = np.average(candidate_earthquakes[name].age, weights=candidate_earthquakes[name].PDF)

#Download candidates dataframe for next part of notebook:
#candidates_df.to_csv("C:/Users/erikc/Downloads/SSAF_to_SanGorgonioPass_Candidates.csv")

Marginal dataframe constructed...


C:\Users\erikc\AppData\Local\Temp\ipykernel_4456\248893994.py:713: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: np.average(x["year"], weights=x["prob"]))


Candidates built...
Current row of new_candidates: 0
Current row of new_candidates: 1
Current row of new_candidates: 2
Current row of new_candidates: 3
Current row of new_candidates: 4
Current row of new_candidates: 5
Current row of new_candidates: 6
Working on above-capacity row...
Current row of new_candidates: 7
Working on above-capacity row...
Current row of new_candidates: 8
Working on above-capacity row...
Current row of new_candidates: 9
Working on above-capacity row...
Current row of new_candidates: 10
Working on above-capacity row...
Current row of new_candidates: 11
Working on above-capacity row...
Current row of new_candidates: 12
Working on above-capacity row...
Current row of new_candidates: 13
Current row of new_candidates: 14
Current row of new_candidates: 15
Current row of new_candidates: 16
Current row of new_candidates: 17
Working on above-capacity row...
Current row of new_candidates: 18
Working on above-capacity row...
Current row of new_candidates: 19
Working on ab

KeyboardInterrupt: 

In [ ]:
import pandas as pd
import itertools as it
import pulp as pl
import numpy as np
import matplotlib.pyplot as plt
import random
import kuehn_et_al_fdm as kea
from kuehn_et_al_fdm import calc_displ_site
from kuehn_et_al_fdm import calc_displ_avg
import gurobipy as gp
import re

fault_length = 386

trenches = {
    "Annette": 20,
    "Bidart": 72,
    "Frazier": 169,
    "Elizabeth": 218,
    "Pallett": 270,
    "Wrightwood": 292,
    "Pitman": 317,
    "Burro": 377,
}

trench_location = list(trenches.values())
trench_names = list(trenches.keys())

#Load candidate events dataframe and drop 'Event_id' column
candidate_events = pd.read_csv("C:/Users/erikc/Downloads/SSAF_to_SanGorgonioPass_Candidates.csv")
candidate_events.drop(columns=candidate_events.columns[-1:], inplace=True)
candidate_events.drop(columns=candidate_events.columns[0], inplace=True)

#Store all trench events as list
trench_events = list(candidate_events)
#Create list of trench names: trench sites without individual event identifiers
trench_keys = [str.split('_', 1)[0] for str in trench_events]

#Group events by trench site

#Create new dictionary to hold the columns of each trench site as its own dataframe
subsets_dict = {}
for i in trenches.keys():
    #isolate the columns of candidate events that correspond to a single trench
    column_group = [k==i for k in trench_keys]
    ce_subset = candidate_events.iloc[:,column_group]
    subsets_dict[i] = ce_subset

#Save a reference array that contains the mean displacements for each magnitude from 6.5 to 8 with resolution down to 0.01 magnitude differences
#Used for magnitude calculation later that is then input into the Kuehn et. al FDM

possible_magnitudes = np.arange(6.5,8.01,0.01)
possible_rupture_length = np.arange(6.5,8.01,0.01)
mean_displacement_array = np.arange(6.5, 8.01, 0.01)
for i in range(len(possible_magnitudes)):
    possible_magnitude = possible_magnitudes[i]
    mean_displacement = kea.calc_displ_avg(
    magnitude = possible_magnitude,
    style = 'strike-slip',
    coefficient_type = 'mean'
)
    mean_displacement_array[i] = mean_displacement
    possible_rupture_length[i] = ((10**(1.5*possible_magnitude + 9.105))/((3*10**10)*(mean_displacement)*15000))

#Store variables used in calculation of magnitude from surface rupture length (from Lavrentiadis et. al, 2023)
B1 = -1.1602
B2 = 0.3395
Wlim = 15 #seismogenic depth for vertically dipping fault with 15km seismogenic thickness - derivation is Dseis / sin(theta) where Dseis = seismogenic thickness and theta = dip angle
Fd = 0 #For San Andreas Fault - see 'A seismologically consistent surface rupture length model for unbounded and width-limited events', Lavrentiadis et. al, 2023
c0 = -3.38783
c1 = 0.9509
c2 = -0.1517

#pip install gurobipy
#print(pl.listSolvers(onlyAvailable=True))

df_list = []
for trench in trench_names:
    indices = []
    for i in range(len(candidates_df.iloc[0])):
        if candidates_df.keys()[i].split('_')[0] == trench:
            indices.append(i)
    df_list.append(candidates_df.iloc[:, np.min(indices) : np.max(indices) + 1].sum(axis = 1))

site_binary = pd.concat(df_list, axis = 1, ignore_index = True)
site_binary_matrix = site_binary.to_numpy()

#New code to calculate oldest possible candidate event that ruptured each trench
#site_oldest_earthquake_age = []
#for site in trench_names:
#    site_oldest_earthquake_age.append(min(marginal.loc[marginal['site'] == site, 'year']))

#Duplicate site_binary_matrix but set sites to 1 if candidate event would have happened before the site's record
prerecord_site_binary_matrix = site_binary_matrix.copy()
for i in range(len(site_binary_matrix)):
    ##New code: calculate oldest possible age of each candidate event
    cols = np.where(candidates_df.iloc[i][0:-1] == 1)[0]
    selected = event_names[cols]
    dfs = [
        marginal[
            (marginal["event_name"] == e) &
                    (marginal["prob"] > threshold)
                ]
            for e in selected
            ]
    mins = [df["year"].min() for df in dfs]
    maxs = [df["year"].max() for df in dfs]

    event_min = max(mins) 
    event_max = min(maxs)
    for j in range(len(site_binary_matrix[0])):
        #If the event occured below the bottom of the trench record, replace 0 with 1 to show earthquake could have ruptured through this trench
        if event_min < trench_oldest_ages[j]:
            prerecord_site_binary_matrix[i][j] = 1
        #If the event occured during a sedimentary hiatus, replace 0 with 1 to show earthquake could have ruptured through this trench
        trench = trench_names[j]
        test = Sedimentary_Hiatus_df.index[(Sedimentary_Hiatus_df['trench'] == trench) & (Sedimentary_Hiatus_df['upper age'] > event_max) & (Sedimentary_Hiatus_df['lower age'] < event_min)]
        if len(test) > 0:
            prerecord_site_binary_matrix[i][j] = 1
        
#Replace row with original row from site binary if there is invalid gap (1s seperated by 0s)
for i in range(len(prerecord_site_binary_matrix)):
    #Ensure there are at least two 1s to have something "in-between"
    if np.count_nonzero(prerecord_site_binary_matrix[i]) > 1:
        #Find the boundary indexes of the first and last '1'
        all_ones = (np.where(prerecord_site_binary_matrix[i] == 1))[0]
        first_one = all_ones[0]
        last_one = all_ones[-1]
        #Check if 0 exists inside this sliced sub-array
        if np.any(prerecord_site_binary_matrix[i][first_one + 1 : last_one] == 0):
            prerecord_site_binary_matrix[i] = site_binary_matrix[i]

l = 20 #20km is arbitrary value to make sure ruptures can extend through apparently 'unruptured' trenches: inequality for unruptured trenches should later filter out endpoints that would cause too much displacement at 'unruptured' trenches

#Find first and last nonzero indices based on prerecord_site_binary_matrix
first_nonzero_index = []
last_nonzero_index = []
for n in range(len(prerecord_site_binary_matrix)):
    first_nonzero_index.append(np.flatnonzero(prerecord_site_binary_matrix[n])[0])
    last_nonzero_index.append(np.flatnonzero(prerecord_site_binary_matrix[n])[-1])

N_Rupture_Endpoint_Max = []
for i in range(len(candidates_df)):
    if first_nonzero_index[i]>0:
        N_Rupture_Endpoint_Max.append(trench_location[first_nonzero_index[i]-1] - l)
#Indexing in above block will screw up the randint generator when first trench is ruptured (index = 0)
    else:
        N_Rupture_Endpoint_Max.append(0)

S_Rupture_Endpoint_Max = []
for i in range(len(candidates_df)):
    if last_nonzero_index[i] < (len(site_binary_matrix[0]) - 1):
        S_Rupture_Endpoint = trench_location[last_nonzero_index[i]+1] + l
        S_Rupture_Endpoint_Max.append(S_Rupture_Endpoint)
#Indexing in above block will screw up the randint generator when last trench is ruptured
    else:
        S_Rupture_Endpoint = fault_length + l
        S_Rupture_Endpoint_Max.append(S_Rupture_Endpoint)

In [ ]:
#Create dataframes for slip rate measurements and slip per event measurements

##IMPORTANT CHOICE: I have chosen to model up to 1812, since that is the oldest historical earthquake. 1812 and
#1857 slip are subtracted off of each slip rate site from the original author's measurement, based on Zielke et al.
#paper. HOWEVER this will tend to lower slip rates for sites that are used as a constant slip rate
#(i.e. Wallace Creek, Gorman Creek, Ranch Center) and should be tested for its effect on the model output

#Create dictionary that holds site name, location, displacement lower bound, displacement upper bound, age lower and upper bounds,
#oldest and youngest possible ages of lower age bound, oldest and youngest possible ages of upper age bound

#IMPORTANT CHOICE: Added 1 m uncertainties on Bidart Fan offset measurements and all measurements where only a single value was given for constraint purposes

###Big Bend lower default slip rate above 10 mm/yr is making model infeasible: for now set to 9 mm/yr


SRS_dictionary = {
    "name": ["Wallace Creek", "Bidart Fan (old)", "Bidart Fan (young)", "Van Matre Ranch", "Gorman Creek", "Oak Flat", "Ranch Center", "Wrightwood", "Cajon Creek", "Pitman Canyon", "Plunge Creek", "Badger Canyon"],
    "location": [66, 72, 72, 84, 176, 198, 238, 292, 314, 348, 317, 374], 
    "displacement lower bound": [20, 9, 3.3, 19.3, 65.8, 36, 56.1, 27, 280, 290, 425, 270],
    "displacement upper bound": [150, 11, 5.3, 21.3, 75.8, 42, 93.1, 57, 300, 510, 875, 300],
    "age lower bound": [-1561, 1418, 1605, 1218, -925, 915, 0, 464, -10588, -32503, -43438, -21288],
    "age upper bound": [1812, 1605, 1812, 1812, 1812, 1812, 1812, 1812, 1812, 1812, 1812, 1812],
    "oldest lower bound": [-1614, 1408, 1600, 1160, -1055, 780, -26, 464, -11588, -34648, -46088, -21688],
    "youngest lower bound": [-1508, 1428, 1610, 1276, -812, 1050, 26, 464, -9588, -30358, -40788, -20888],
    "oldest upper bound": [1812, 1600, 1812, 1812, 1812, 1812, 1812, 1812, 1812, 1812, 1812, 1812],
    "youngest upper bound": [1812, 1610, 1812, 1812, 1812, 1812, 1812, 1812, 1812, 1812, 1812, 1812],
}

SRS_df = pd.DataFrame(SRS_dictionary)

#*IMPORTANT NOTE: Disagreement in 1812 + 1857 slip in Wrightwood vicinity. Salyards et al. (1992) estimated
#8.5 +- 1 m total combined slip at Pallett Creek, whereas Zielke et al. (2012) estimated 3.1 m total combined slip at Wrightwood
#WILL NEED TO TEST RANGE OF VALUES BETWEEN 3.1 AND 9.5 m

#**NOTE: slip rate sites whose upper bound is present have had historical slip (1812 and 1857) subtracted off their total displacement

#Wrightwood is probably missing an event between 1300 and 1400, per https://rock.geosociety.org/net/gsatoday/archive/14/9/pdf/i1052-5173-14-9-4.pdf (also good source for 1812 displacements)

#Wallace Creek should be used as a lower and upper bound (for range of values from 35 - 37mm/yr)
#Bidart Fan displacement sets should be used to optimize against, 
#Gorman Creek should be used as a lower bound, 
#Oak Flat should be used as a lower and upper bound to optimize against (34 - 51mm/yr), 
#Ranch Center should be used as a lower and upper bound to optimize against (~30 - 49mm/yr)
#Pallett Creek displacement sets should be used to optimize against
#Wrightwood should be optimized against
#Plunge Creek should be used as a lower and upper bound
#Badger Canyon should be used to optimize against
#Pitman Canyon should be used to optimize against
#Cajon Creek should be used to optimize against
#Van Matre Ranch should be used to optimize against

SRS_site_names = np.array(SRS_df["name"])

#Input names of sites you want to OPTIMIZE WITH here:
optimize_site_names = ["Bidart Fan (old)", "Bidart Fan (young)", "Van Matre Ranch", "Oak Flat", "Ranch Center", "Wrightwood", "Badger Canyon", "Pitman Canyon", "Cajon Creek"]

#Input names of sites you want to CONSTRAIN (LOWER AND UPPER BOUND) WITH here:
constrain_site_names = ["Van Matre Ranch", "Wallace Creek", "Bidart Fan (old)", "Bidart Fan (young)", "Oak Flat", "Ranch Center", "Wrightwood", "Plunge Creek", "Badger Canyon", "Pitman Canyon", "Cajon Creek", "Van Matre Ranch"]

#Input names of sites you want to CONSTRAIN (LOWER BOUND) WITH here:
constrain_lower_bd_site_names = ["Gorman Creek"]

#Input names of sites you want to CONSTRAIN (UPPER BOUND) WITH here:
constrain_upper_bd_site_names = []

##Use after extending fault model
#Set default slip rate constraints at intervals along full length of fault

##Test: lower and raise default slip rates
Big_Bend_default_low_rate = 0.009 #20 - 51 mm/yr for 'Big Bend' section, based on SCEC Geologic slip rate minimum and maximum rates along section
Big_Bend_default_high_rate = 0.051

San_Bernadino_default_low_rate = 0.002 #2 - 28 mm/yr for 'San Bernadino' section (i.e. between Cajon and San Gorgonio passes), again based on the maximum and minimum rates in the SCEC Slip Rate Database
San_Bernadino_default_high_rate = 0.028

#Slip-per-event data: constrain model based on estimates of single-event slip from trenches

#Frazier_vertical_displacements = [0.8, 0.85, 0.2, 1, 0.25 - 0.6, 0.27, 0.48, 0.27, 0 - 1, 0 - 1]
#Pallett_brittle_displacements = [0.8 - 1.5*, 1 - 1.9*, 0.5 - 1, 0.1 - 0.2, 0.1 - 0.15, 2, 1 - 3, 1 - 3]  

# *summed slip from these first two events has to be between 2.5m - 2.7m
#Pallett Event X not included because it is the 1812 historical San Andreas earthquake (set sum of decision variables in associated candidates table == 0)
#Pallett_Event_I upper bound raised from 0.73 to 1.2 m since it was making model unsolvable in SPE_constrain mode

#Using scalar factor calculated from vertical to horizontal ratio for Fort Tejon Earthquake
#for Annette site yields MINIMUM slip rate of 69 mm/yr; therefore, I do not include SPE for Annette site

#Choose events whose summed displacement is better constrained
summed_d_events = [['Pallett_Event_V', 'Pallett_Event_T'], ['Burro_Event_3', 'Burro_Event_4'], [['Pitman_Event_2', 'Pitman_Event_3', 'Pitman_Event_4']]
summed_d_mean = [[6.9], [8], [12]]
summed_d_lower_bd = [[1.5], [7], [10]]
summed_d_upper_bd = [[10.5], [9], [14]]

SPE_dictionary = {
    "event": ['Frazier_Event_2', 'Frazier_Event_3', 'Frazier_Event_4', 'Frazier_Event_5', 'Frazier_Event_6', 'Frazier_Event_7', 'Frazier_Event_8', 'Frazier_Event_9', 'Frazier_Event_10', 'Frazier_Event_11',
              'Pallett_Event_V', 'Pallett_Event_T', 'Pallett_Event_R', 'Pallett_Event_N', 'Pallett_Event_I', 'Pallett_Event_F', 'Pallett_Event_D', 'Pallett_Event_C',
              'Wrightwood_Event_W3', 'Wrightwood_Event_W4', 'Wrightwood_Event_W5', 'Wrightwood_Event_W6', 'Wrightwood_Event_W7', 'Wrightwood_Event_W8', 'Wrightwood_Event_W9', 'Wrightwood_Event_W10', 'Wrightwood_Event_W11', 'Wrightwood_Event_W12', 'Wrightwood_Event_W13', 'Wrightwood_Event_W14',
              'Pitman_Event_2',
              'Burro_Event_2'],
             
    "lower_bd": [2.2, 2.4, 0.56, 2.8, 0.7, 0.75, 1.3, 0.75, 0.56, 0.56,
                 2.5, 1.60, 0.80, 0.16, 0.16, 3.2, 1.6, 1.6,
                 1.0, 3.0, 0.0, 1.9, 1.1, 0.7, 3.0, 2.5, 1.2, 1.4, 1.0, 1.0,
                 3.0,
                 3.0],
                
    "upper_bd": [7.9, 8.4, 2.0, 9.9, 5.9, 2.7, 4.8, 2.7, 9.9, 9.9,
                 8.5, 9.3, 4.9, 0.98, 1.2, 9.8, 15, 15,
                 7.0, 9.0, 2.8, 5.6, 3.4, 3.1, 9.9, 7.5, 6.3, 8.2, 5.1, 3.8,
                 5.0,
                 5.0],
                
    "mean": [4.9, 5.2, 1.2, 6.1, 2.6, 1.7, 2.9, 1.7, 3.1, 3.1,
             5.5, 4.7, 2.9, 0.55, 0.47, 7.3, 7.3, 7.3, 
             3.5, 7.0, 0.7, 3.7, 1.8, 1.5, 6.6, 5.2, 3.0, 4.1, 1.8, 1.9,
             4.0,
             4.0]
            
}

SPE_df = pd.DataFrame(SPE_dictionary)

#Used Frazier Mountain SPE to estimate dextral slip per event (using best fitting scalar factor between vertical and
#horizontal slip of 6.14 based on Zielke and Arrowsmith's 1857
#Fort Tejon Earthquake slip profile giving ~4.3m of slip at Frazier Mountain, which caused an observed 0.7m of
#vertical slip at Frazier Mountain to get a slip rate of ~25.9 mm/yr, maximum scalar factor of 9.90 to get a slip rate of 51 mm/yr,
#minimum scalar factor of 2.78 to get a slip rate of 10 mm/yr)

#Used Pallett Creek SPE to estimate total (brittle + nonbrittle) dextral slip per event (using best fitting scalar factor of
#2.64 based on Salyard and Sieh's estimate of 14 m of nonbrittle deformation caused by ~5.3 m of brittle deformation
#during 3 most recent earthquakes at Pallett Creek (Events Z, X, V) to get a slip rate of 30.6 mm/yr (using mean SPEs)), 
#maximum scalar factor of 3.88 to get a slip rate of 51 mm/yr, minimum scalar factor of 0.601 to get a slip rate of 10 mm/yr


#Put events in order of increasing size from left -> right
SPE_relative_list = [
    ['Annette_Event_2', 'Annette_Event_3', 'Annette_Event_6', 'Annette_Event_4', 'Annette_Event_5'],
    ['Bidart_Event_3', 'Bidart_Event_4'],
    ['Frazier_Event_4', 'Frazier_Event_7', 'Frazier_Event_6', 'Frazier_Event_8', 'Frazier_Event_10', 'Frazier_Event_2', 'Frazier_Event_3', 'Frazier_Event_5'],
    ['Pallett_Event_I', 'Pallett_Event_N', 'Pallett_Event_R', 'Pallett_Event_T', 'Pallett_Event_V'],
    ['Wrightwood_Event_W5', 'Wrightwood_Event_W8', 'Wrightwood_Event_W14', 'Wrightwood_Event_W11', 'Wrightwood_Event_W3', 'Wrightwood_Event_W6', 'Wrightwood_Event_W12', 'Wrightwood_Event_W10', 'Wrightwood_Event_W9', 'Wrightwood_Event_W4']
]

#Put same size events in their own bracketed ([ ]) groups
equal_size_events = [['Frazier_Event_7', 'Frazier_Event_9'], ['Frazier_Event_10', 'Frazier_Event_11'], ['Pallett_Event_C', 'Pallett_Event_D', 'Pallett_Event_F'], ['Wrightwood_Event_W7', 'Wrightwood_Event_W13']]

In [ ]:
#Calculate matrix of displacements for each site for each candidate event: follows sequence (Candidate1Site1,Candidate1Site2,Candidate1Site3,Candidate1Site4,Candidate2Site1,Candidate2Site2...)
#To be used only for observed/unobserved displacement constraint in ILP Model
#(runtime ~ 22 minutes at t = 15km resolution and 20 km constraint spacing)

t = 15 #t km endpoint intervals

default_slip_rate_constraint_spacing = 40 #Spacing of points (km) to constrain absolute lower and upper bound of possible slip rate

SRL_threshold = 45 #Minimum surface rupture length (km) to possibly create an observable displacement (NOTE: should be 47 km according to Kuehn et al. model, but doing so makes all single-site events at Pallett Creek have 0 displacement due to spacing of endpoints and proximity to other trenches)

#Create displacement matrices for each trench (to be used to constrain observed/unobserved displacements at each site
#as well as slip-per-event (SPE) optimization and constraints)

trench_site_names = list(trenches.keys())

trench_displacements = [[[] for i in range(len(trench_location))] for j in range(len(site_binary_matrix))]

for x in range(len(trench_location)):                     #loop "columnwise", for each paleoseismic site individually
    for n in range(len(site_binary_matrix)):              #loop for each candidate event
        Possible_N_Endpoints_Array = np.arange(N_Rupture_Endpoint_Max[n], S_Rupture_Endpoint_Max[n], t)
        Possible_S_Endpoints_Array = np.arange(N_Rupture_Endpoint_Max[n], S_Rupture_Endpoint_Max[n], t)
        for i in range(len(Possible_S_Endpoints_Array)):  #Loop over all possible S Endpoints based on ruptured/unruptred constraints
            for j in range(i): #Loop over all possible N Endpoints for given S Endpoint based on S > N constraint
                #If the trench location is outside the rupture OR the rupture length is smaller than M6.5, set to 0
                if trench_location[x] >= Possible_S_Endpoints_Array[i] or trench_location[x] <= Possible_N_Endpoints_Array[j] or Possible_S_Endpoints_Array[i] - Possible_N_Endpoints_Array[j] < SRL_threshold:
                    Displacement = 0
                    trench_displacements[n][x].append(Displacement)
                else:
                    rupture_l = Possible_S_Endpoints_Array[i] - Possible_N_Endpoints_Array[j]
                    index = (np.abs(possible_rupture_length/1000 - (rupture_l))).argmin()
                    magnitude = (index)*0.01 + 6.5
                    Displacement = kea.calc_displ_site(
                    magnitude = magnitude,
                    location = (Possible_S_Endpoints_Array[i] - trench_location[x])/(rupture_l), #KNOB: assumes earthquakes are propagating northwards
                    style = 'strike-slip', 
                    percentile = -1 #KNOB: using mean (-1) for percentile, will later test different percentiles of slip
                    ) 
                    trench_displacements[n][x].append(Displacement)

#Create displacements matrix for each slip rate site for each candidate event

srs_displacements = [[[] for i in range(len(SRS_df['name']))] for j in range(len(site_binary_matrix))]

for x in range(len(SRS_df['name'])):                     #loop "columnwise", for each paleoseismic site individually
    for n in range(len(site_binary_matrix)):              #loop for each candidate event
        Possible_N_Endpoints_Array = np.arange(N_Rupture_Endpoint_Max[n], S_Rupture_Endpoint_Max[n], t)
        Possible_S_Endpoints_Array = np.arange(N_Rupture_Endpoint_Max[n], S_Rupture_Endpoint_Max[n], t)
        for i in range(len(Possible_S_Endpoints_Array)):  #Loop over all possible S Endpoints based on ruptured/unruptred constraints
            for j in range(i):                            #Loop over all possible N Endpoints for given S Endpoint based on S > N constraint
                if SRS_df['location'][x] >= Possible_S_Endpoints_Array[i] or SRS_df['location'][x] <= Possible_N_Endpoints_Array[j] or Possible_S_Endpoints_Array[i] - Possible_N_Endpoints_Array[j] < SRL_threshold:
                    Displacement = 0
                    srs_displacements[n][x].append(Displacement)
                else:
                    rupture_l = Possible_S_Endpoints_Array[i] - Possible_N_Endpoints_Array[j]
                    index = (np.abs(possible_rupture_length/1000 - (rupture_l))).argmin()
                    magnitude = (index)*0.01 + 6.5
                    Displacement = kea.calc_displ_site(
                    magnitude = magnitude,
                    location = (Possible_S_Endpoints_Array[i] - SRS_df['location'][x])/(rupture_l), #KNOB: assumes earthquakes are propagating northwards
                    style = 'strike-slip', 
                    percentile = -1 #KNOB: using mean (-1) for percentile, will later test different percentiles of slip
                    ) 
                    srs_displacements[n][x].append(Displacement)

#Create displacements matrix for points uniformly spaced along fault to constrain slip rate

default_slip_rate_locations = np.arange(trench_location[0], trench_location[-1], default_slip_rate_constraint_spacing)

default_displacements = [[[] for i in range(len(default_slip_rate_locations))] for j in range(len(site_binary_matrix))]

for x in range(len(default_slip_rate_locations)):                     #loop "columnwise", for each paleoseismic site individually
    for n in range(len(site_binary_matrix)):              #loop for each candidate event
        Possible_N_Endpoints_Array = np.arange(N_Rupture_Endpoint_Max[n], S_Rupture_Endpoint_Max[n], t)
        Possible_S_Endpoints_Array = np.arange(N_Rupture_Endpoint_Max[n], S_Rupture_Endpoint_Max[n], t)
        for i in range(len(Possible_S_Endpoints_Array)):  #Loop over all possible S Endpoints based on ruptured/unruptred constraints
            for j in range(i):                            #Loop over all possible N Endpoints for given S Endpoint based on S > N constraint
                if default_slip_rate_locations[x] >= Possible_S_Endpoints_Array[i] or default_slip_rate_locations[x] <= Possible_N_Endpoints_Array[j] or Possible_S_Endpoints_Array[i] - Possible_N_Endpoints_Array[j] < SRL_threshold:
                    Displacement = 0
                    default_displacements[n][x].append(Displacement)
                else:
                    rupture_l = Possible_S_Endpoints_Array[i] - Possible_N_Endpoints_Array[j]
                    index = (np.abs(possible_rupture_length/1000 - (rupture_l))).argmin()
                    magnitude = (index)*0.01 + 6.5
                    Displacement = kea.calc_displ_site(
                    magnitude = magnitude,
                    location = (Possible_S_Endpoints_Array[i] - default_slip_rate_locations[x])/(rupture_l), #KNOB: assumes earthquakes are propagating northwards
                    style = 'strike-slip', 
                    percentile = -1 #KNOB: using mean (-1) for percentile, will later test different percentiles of slip
                    ) 
                    default_displacements[n][x].append(Displacement)
#Precompute indices of ruptured vs. unruptured trenches for each candidate event
#Also precompute indices of displacements from 'displacements' that are above and below threshold (currently set to 0.5m for observation)
#Make displacements array uniform so all values are 0D numpy arrays

threshold = 0.5 #KNOB: change the minimum required displacement (in meters) for event to be observed/unobserved in trench

for n in range(len(site_binary_matrix)):
    for site in range(len(site_binary_matrix[0])):
        for i in range(len(trench_displacements[n][site])):
            val = trench_displacements[n][site][i]
            if isinstance(val, np.ndarray):
                trench_displacements[n][site][i] = float(val[0])

ruptured = [[[] for site in range(len(site_binary_matrix[0]))] for n in range(len(site_binary_matrix))]
unruptured = [[[] for site in range(len(site_binary_matrix[0]))] for n in range(len(site_binary_matrix))]
for n in range(len(site_binary_matrix)):
    for site in range(len(site_binary_matrix[0])):
        ruptured[n][site] = np.where(np.array(trench_displacements[n][site]) > threshold)[0]
        unruptured[n][site] = np.where(np.array(trench_displacements[n][site]) < threshold)[0]

#Store endpoints used to calculate displacements in their own seperate arrays (to be indexed later to find actual endpoint values once decision variables are chosen)
#Follows same matrix-derived pattern as above, but only one set (one array) for each candidate event (each site displacement matrix is calculated using the same matrix of endpoints)

S = [[] for n in range(len(site_binary_matrix))]
N = [[] for n in range(len(site_binary_matrix))]



for n in range(len(site_binary_matrix)):  
    for i in range(int((S_Rupture_Endpoint_Max[n] - N_Rupture_Endpoint_Max[n])/t)+1):
        for j in range(i):
            endpoint = N_Rupture_Endpoint_Max[n] + t*i
            S[n].append(endpoint)

for n in range(len(site_binary_matrix)):  
    for i in range(int((S_Rupture_Endpoint_Max[n] - N_Rupture_Endpoint_Max[n])/t)+1):
        for j in range(i):
            endpoint = N_Rupture_Endpoint_Max[n] + t*j
            N[n].append(endpoint)

#Store the total number of possible endpoint combinations generated for each candidate event in its own array
candidate_displacements_count = []
for n in range(len(site_binary_matrix)):
    count = len(trench_displacements[n][0])
    candidate_displacements_count.append(count)

In [ ]:
#Precalculate constraints for ILP Model

#Create model
model = pl.LpProblem("Plausible_Event_Set", pl.LpMinimize)

#Index set
E = candidates_df.index.tolist()

#Create n binary decision variables for each candidate event, each representing a different combination of rupture endpoints (n = number of valid combinations)

y = []
for n in range(len(site_binary_matrix)):
    var_set = pl.LpVariable.dicts(f"y_{n}", range(len(trench_displacements[n][0])), cat='Binary')
    y.append(var_set)

#Constraint: trench events stay in stratigraphic order

def natural_sort_key(s):
    return [
        int(text) if text.isdigit() else text.lower()
        for text in re.split(r'(\d+)', s)
    ]

for site in trench_names:

    event_names = []

    for event in event_keep_names:
        if event.split('_')[0] == site:
            event_names.append(event)
    #Add special condition for Pallett Creek because the oldest event is the earliest letter in the alphabet
    if site == 'Pallett':
        event_names_sorted = ['Pallett_Event_V', 'Pallett_Event_T', 'Pallett_Event_R', 'Pallett_Event_N', 'Pallett_Event_I', 'Pallett_Event_F', 'Pallett_Event_D', 'Pallett_Event_C']
   #Add special condition for Elizabeth Lake because Event X is chronologically first but falls last when sorted
    elif site == 'Elizabeth':
        event_names_sorted = ['Elizabeth_Event_ELX', 'Elizabeth_Event_EL2', 'Elizabeth_Event_EL3', 'Elizabeth_Event_EL4']
    else:
        event_names_sorted = sorted(event_names, key=natural_sort_key)

    #Compare each event with the event immediately below it
    for j in range(len(event_names_sorted) - 1):

        name1 = event_names_sorted[j]
        name2 = event_names_sorted[j + 1]

        #Find the candidate earthquake indices belonging to each event
        group1 = candidates_df.index[
            candidates_df[name1] == 1
        ]

        group2 = candidates_df.index[
            candidates_df[name2] == 1
        ]

        #Multiply each event's modeled age for that candidate event by its decision variable for that event
        age_event1 = pl.lpSum(
            candidate_earthquake_ages[n] * y[n][i]
            for n in group1
            for i in range(len(y[n]))
        )
 
        age_event2 = pl.lpSum(
            candidate_earthquake_ages[n] * y[n][i]
            for n in group2
            for i in range(len(y[n]))
        )

        #Must use greater than or equal to for continuous optimization
        model += age_event1 >= age_event2

#Constraint: One event per column (each trench event must belong to only one candidate event)

site_event_list = candidates_df.columns

#Replaced x with y for '1 event per column' constraint
for col in site_event_list:
    group = candidates_df[candidates_df[col] == 1].index
    if len(group) > 0:
        model += pl.lpSum(y[n][i] for n in group for i in range(len(y[n]))) == 1

#Constraint: Pick AT MOST one endpoint from a candidate event's displacement set (may be redundant, not tested)
for n in E:
    model += pl.lpSum(y[n][i] for i in range(len(y[n]))) <= 1

#Constraint: If trench showed 'ruptured' (1), displacement must be greater than 0.5 (and vice versa)

#Go through all the displacement subarrays that correspond to trenches that are ruptured (and unruptured) and force the y[n][i] that correspond to
#displacements in these subarrays that are not consistent with either being observed or not observed to be 0

#ILP Model 4.0: Remove displacement constraints on sites that are in candidate events that occured before the site's records began
#Replaced >= age_limit with >= site_oldest... because age_limit is sometimes younger than oldest event at site after 'stringing pearls'
for n in range(len(site_binary_matrix)):
    for site in range(len(site_binary_matrix[0])):
        if candidate_earthquake_ages[n] >= trench_oldest_ages[site]:
            if site_binary_matrix[n][site] == 1: #Add constraint that if there is a 1 in site's position in site_binary, all y's that correspond to displacements less than threshold must equal zero
                for index in unruptured[n][site]:
                    model += y[n][index] == 0
            if site_binary_matrix[n][site] == 0:
                displacements = np.array(trench_displacements[n][site], dtype=object)
                indices = np.where(displacements > threshold)[0]
                for index in indices:
                    model += y[n][index] == 0

In [ ]:
#ILP Model

#Settings:

#Choose whether to turn on slip rate optimization
SRS_optimize = 'on'

#If SRS_optimize on, choose slip rate scenario: 'max', 'min', or 'mean' (max = use upper bound on displacement or slip rate for each site, min = use lower bound on displacement or slip rate for each site)
SRS_scenario = 'mean'

#Choose whether to turn on slip-per-event constraints, slip-per-event optimization, or neither
SPE_constrain = 'off'
SPE_optimize = 'off'

#If SPE_optimize on, choose slip-per-event scenario: 'max', 'min', or 'mean' (max = use upper bound on slip per event, min = use lower bound on slip per event)
SPE_scenario = 'mean'

#Choose whether to constrain event size based on estimates of event slip relative to other events at the same site
SPE_relative_constrain = 'on'

equality_threshold = 1 #Maximum allowed difference in lateral slip (in meters) to be allowed to be 'relatively' the same sized event


##(former position of ILP Constraints)

#Modeled Slip using endpoint decision variables and displacements matrix (all sites for a candidate event have same dimension of y[n], which is the same dimension as displacements[n][site]):

#Slip rate data: if slip rate data lower bound is younger than its two adjacent paleoseismic sites, use its displacement + time interval
#if slip rate data lower bound is older than its two adjacent paleoseismic sites, only use the calculated rate x time interval between oldest adjacent earthquake and 1812/1857

#Create objective functions for each SRS that was chosen to optimize with

residual = pl.LpAffineExpression()

#Replaced age_limits everywhere below with trench_oldest_ages, since using age_limits means there's
#a good chance we aren't constraining all the events that involve those trenches. However, this does add some risk
#that we are constraining an event that ruptured through before the oldest event at a site (for a given scenario)

#MEAN slip scenario
if SRS_optimize == 'on':
    if SRS_scenario == 'mean':
        for site in optimize_site_names:
            r = pl.LpVariable(f"SRS_r_{site}", lowBound=0) #Create residual variable for each slip rate site to optimize with
            index = np.where(SRS_site_names == site)[0][0]
            position = SRS_df.loc[SRS_df['name'] == site, 'location']
            N_trench_index = np.searchsorted(trench_location, position) - 1 #Find the indexes of the two adjacent trenches
            S_trench_index = N_trench_index + 1
        #For this study, I have chosen to use the youngest of the two adjacent paleoseismic site's oldest ages,
        #to make sure that the displacement is constrained using ALL the earthquakes that have ruptured through
        #the slip rate site. Another possibility is to use min() instead, since we have dealt with the censoring problem
            if isinstance(N_trench_index, np.int64): #Handles bug with N_trench_index alternating data types
                adjacent_sites_age_limit = max(int(trench_oldest_ages[N_trench_index]), int(trench_oldest_ages[S_trench_index]))
            else:
                adjacent_sites_age_limit = max(int(trench_oldest_ages[N_trench_index[0]]), int(trench_oldest_ages[S_trench_index[0]]))
            lower_bound = (SRS_df.loc[SRS_df['name'] == site, 'age lower bound']).item()
            #Set condition to make sure the events we are constraining happened when we have paleoseismic records on both sides
            if lower_bound <= adjacent_sites_age_limit:
                lower_bound = adjacent_sites_age_limit
            upper_bound = SRS_df.loc[SRS_df['name'] == site, 'age upper bound'].item()
            constrained_event_indices = np.where(
                (candidate_earthquake_ages >= lower_bound) & 
                (candidate_earthquake_ages <= upper_bound)
            )[0]
            #If lower age on displacement measurement is within paleoseismic record, constrain events in that time period to that displacement
            if (SRS_df.loc[SRS_df['name'] == site, 'age lower bound']).item() > adjacent_sites_age_limit - 15: #Add 15 year buffer (for Bidart)
                model += r >= ((SRS_df.loc[SRS_df['name'] == site, 'displacement lower bound'] + SRS_df.loc[SRS_df['name'] == site, 'displacement upper bound'])/2) - pl.lpSum(srs_displacements[n][index][i] * y[n][i] for n in constrained_event_indices for i in range(len(srs_displacements[n][0])))
                model += r >= -(((SRS_df.loc[SRS_df['name'] == site, 'displacement lower bound'] + SRS_df.loc[SRS_df['name'] == site, 'displacement upper bound'])/2) - pl.lpSum(srs_displacements[n][index][i] * y[n][i] for n in constrained_event_indices for i in range(len(srs_displacements[n][0]))))
                residual += r
            #If lower age on displacement measurement is pre-record, turn it into a slip rate to optimize against
            else:
                slip_rate = ((SRS_df.loc[SRS_df['name'] == site, 'displacement lower bound'] + SRS_df.loc[SRS_df['name'] == site, 'displacement upper bound'])/2)/(SRS_df.loc[SRS_df['name'] == site, 'age upper bound'] - SRS_df.loc[SRS_df['name'] == site, 'age lower bound'])
                model += r >= slip_rate*(upper_bound - lower_bound) - pl.lpSum(srs_displacements[n][index][i] * y[n][i] for n in constrained_event_indices for i in range(len(srs_displacements[n][0])))
                model += r >= -(slip_rate*(upper_bound - lower_bound) - pl.lpSum(srs_displacements[n][index][i] * y[n][i] for n in constrained_event_indices for i in range(len(srs_displacements[n][0]))))
                residual += r

#MAX slip scenario

    if SRS_scenario == 'max':
        for site in optimize_site_names:
            r = pl.LpVariable(f"SRS_r_{site}", lowBound=0) #Create residual variable for each slip rate site to optimize with
            index = np.where(SRS_site_names == site)[0][0]
            position = SRS_df.loc[SRS_df['name'] == site, 'location']
            N_trench_index = np.searchsorted(trench_location, position) - 1
            S_trench_index = N_trench_index + 1
        #For this study, I have chosen to use the youngest of the two adjacent paleoseismic site's oldest ages,
        #to make sure that the displacement is constrained using ALL the earthquakes that have ruptured through
        #the slip rate site. Another possibility is to use min() instead, since we have dealt with the censoring problem
            if isinstance(N_trench_index, np.int64):
                adjacent_sites_age_limit = max(int(trench_oldest_ages[N_trench_index]), int(trench_oldest_ages[S_trench_index]))
            else:
                adjacent_sites_age_limit = max(int(trench_oldest_ages[N_trench_index[0]]), int(trench_oldest_ages[S_trench_index[0]]))
            lower_bound = (SRS_df.loc[SRS_df['name'] == site, 'age lower bound']).item()
        #Set condition to make sure the events we are constraining happened when we have paleoseismic records on both sides
            if lower_bound <= adjacent_sites_age_limit:
                lower_bound = adjacent_sites_age_limit
            upper_bound = SRS_df.loc[SRS_df['name'] == site, 'age upper bound'].item()
            constrained_event_indices = np.where(
                (candidate_earthquake_ages >= lower_bound) & 
                (candidate_earthquake_ages <= upper_bound)
            )[0]
            if (SRS_df.loc[SRS_df['name'] == site, 'age lower bound']).item() > adjacent_sites_age_limit - 15: #Add 15 year buffer (for Bidart)
                model += r >= SRS_df.loc[SRS_df['name'] == site, 'displacement upper bound'] - pl.lpSum(srs_displacements[n][index][i] * y[n][i] for n in constrained_event_indices for i in range(len(srs_displacements[n][0])))
                model += r >= -(SRS_df.loc[SRS_df['name'] == site, 'displacement upper bound'] - pl.lpSum(srs_displacements[n][index][i] * y[n][i] for n in constrained_event_indices for i in range(len(srs_displacements[n][0]))))
                residual += r
            #If lower age on displacement measurement is pre-record, turn it into a slip rate to optimize against
            else:
                slip_rate = (SRS_df.loc[SRS_df['name'] == site, 'displacement upper bound'])/(SRS_df.loc[SRS_df['name'] == site, 'oldest upper bound'] - SRS_df.loc[SRS_df['name'] == site, 'youngest lower bound'])
                model += r >= slip_rate*(upper_bound - lower_bound) - pl.lpSum(srs_displacements[n][index][i] * y[n][i] for n in constrained_event_indices for i in range(len(srs_displacements[n][0])))
                model += r >= -(slip_rate*(upper_bound - lower_bound) - pl.lpSum(srs_displacements[n][index][i] * y[n][i] for n in constrained_event_indices for i in range(len(srs_displacements[n][0]))))
                residual += r

#MIN slip scenario

    if SRS_scenario == 'min':
        for site in optimize_site_names:
            r = pl.LpVariable(f"SRS_r_{site}", lowBound=0) #Create residual variable for each slip rate site to optimize with
            index = np.where(SRS_site_names == site)[0][0]
            position = SRS_df.loc[SRS_df['name'] == site, 'location']
            N_trench_index = np.searchsorted(trench_location, position) - 1
            S_trench_index = N_trench_index + 1
        #For this study, I have chosen to use the youngest of the two adjacent paleoseismic site's oldest ages,
        #to make sure that the displacement is constrained using ALL the earthquakes that have ruptured through
        #the slip rate site. Another possibility is to use min() instead, since we have dealt with the censoring problem
            if isinstance(N_trench_index, np.int64):
                adjacent_sites_age_limit = max(int(trench_oldest_ages[N_trench_index]), int(trench_oldest_ages[S_trench_index]))
            else:
                adjacent_sites_age_limit = max(int(trench_oldest_ages[N_trench_index[0]]), int(trench_oldest_ages[S_trench_index[0]]))
            lower_bound = (SRS_df.loc[SRS_df['name'] == site, 'age lower bound']).item()
        #Set condition to make sure the events we are constraining happened when we have paleoseismic records on both sides
            if lower_bound <= adjacent_sites_age_limit:
                lower_bound = adjacent_sites_age_limit
            upper_bound = SRS_df.loc[SRS_df['name'] == site, 'age upper bound'].item()
            constrained_event_indices = np.where(
                (candidate_earthquake_ages >= lower_bound) & 
                (candidate_earthquake_ages <= upper_bound)
            )[0]
            if (SRS_df.loc[SRS_df['name'] == site, 'age lower bound']).item() > adjacent_sites_age_limit - 15: #Add 15 year buffer (for Bidart)
                model += r >= SRS_df.loc[SRS_df['name'] == site, 'displacement lower bound'] - pl.lpSum(srs_displacements[n][index][i] * y[n][i] for n in constrained_event_indices for i in range(len(srs_displacements[n][0])))
                model += r >= -(SRS_df.loc[SRS_df['name'] == site, 'displacement lower bound'] - pl.lpSum(srs_displacements[n][index][i] * y[n][i] for n in constrained_event_indices for i in range(len(srs_displacements[n][0]))))
                residual += r
            #If lower age on displacement measurement is pre-record, turn it into a slip rate to optimize against
            else:
                slip_rate = (SRS_df.loc[SRS_df['name'] == site, 'displacement lower bound'])/(SRS_df.loc[SRS_df['name'] == site, 'youngest upper bound'] - SRS_df.loc[SRS_df['name'] == site, 'oldest lower bound'])
                model += r >= slip_rate*(upper_bound - lower_bound) - pl.lpSum(srs_displacements[n][index][i] * y[n][i] for n in constrained_event_indices for i in range(len(srs_displacements[n][0])))
                model += r >= -(slip_rate*(upper_bound - lower_bound) - pl.lpSum(srs_displacements[n][index][i] * y[n][i] for n in constrained_event_indices for i in range(len(srs_displacements[n][0]))))
                residual += r

#Create constraints for each SRS that was chosen to constrain (upper and lower bound) with

for site in constrain_site_names:
    index = np.where(SRS_site_names == site)[0][0]
    position = SRS_df.loc[SRS_df['name'] == site, 'location']
    N_trench_index = np.searchsorted(trench_location, position) - 1
    S_trench_index = N_trench_index + 1
    if isinstance(N_trench_index, np.int64):
        adjacent_sites_age_limit = max(int(trench_oldest_ages[N_trench_index]), int(trench_oldest_ages[S_trench_index]))
    else:
        adjacent_sites_age_limit = max(int(trench_oldest_ages[N_trench_index[0]]), int(trench_oldest_ages[S_trench_index[0]]))        
    lower_bound = (SRS_df.loc[SRS_df['name'] == site, 'age lower bound']).item()
    #Set condition to make sure the events we are constraining happened when we have paleoseismic records on both sides
    if lower_bound <= adjacent_sites_age_limit:
        lower_bound = adjacent_sites_age_limit
    upper_bound = SRS_df.loc[SRS_df['name'] == site, 'age upper bound'].item()
    constrained_event_indices = np.where(
        (candidate_earthquake_ages >= lower_bound) & 
        (candidate_earthquake_ages <= upper_bound)
    )[0]
    if (SRS_df.loc[SRS_df['name'] == site, 'age lower bound']).item() > adjacent_sites_age_limit - 15:
        model += SRS_df.loc[SRS_df['name'] == site, 'displacement lower bound'].item() <= pl.lpSum(srs_displacements[n][index][i] * y[n][i] for n in constrained_event_indices for i in range(len(srs_displacements[n][0]))) <= SRS_df.loc[SRS_df['name'] == site, 'displacement upper bound'].item()
    else:
        low_slip_rate = (SRS_df.loc[SRS_df['name'] == site, 'displacement lower bound'])/(SRS_df.loc[SRS_df['name'] == site, 'youngest upper bound'] - SRS_df.loc[SRS_df['name'] == site, 'oldest lower bound'])
        high_slip_rate = (SRS_df.loc[SRS_df['name'] == site, 'displacement upper bound'])/(SRS_df.loc[SRS_df['name'] == site, 'oldest upper bound'] - SRS_df.loc[SRS_df['name'] == site, 'youngest lower bound'])
        model += (low_slip_rate.item()*(upper_bound - lower_bound)) <= pl.lpSum(srs_displacements[n][index][i] * y[n][i] for n in constrained_event_indices for i in range(len(srs_displacements[n][0]))) <= (high_slip_rate.item()*(upper_bound - lower_bound))

#Create constraints for each SRS that was chosen to constrain (lower bound) with

for site in constrain_lower_bd_site_names:
    index = np.where(SRS_site_names == site)[0][0]
    position = SRS_df.loc[SRS_df['name'] == site, 'location']
    N_trench_index = np.searchsorted(trench_location, position) - 1
    S_trench_index = N_trench_index + 1
    if isinstance(N_trench_index, np.int64):
        adjacent_sites_age_limit = max(int(trench_oldest_ages[N_trench_index]), int(trench_oldest_ages[S_trench_index]))
    else:
        adjacent_sites_age_limit = max(int(trench_oldest_ages[N_trench_index[0]]), int(trench_oldest_ages[S_trench_index[0]]))        
    lower_bound = (SRS_df.loc[SRS_df['name'] == site, 'age lower bound']).item()
    #Set condition to make sure the events we are constraining happened when we have paleoseismic records on both sides
    if lower_bound <= adjacent_sites_age_limit:
        lower_bound = adjacent_sites_age_limit
    upper_bound = SRS_df.loc[SRS_df['name'] == site, 'age upper bound'].item()
    constrained_event_indices = np.where(
        (candidate_earthquake_ages >= lower_bound) & 
        (candidate_earthquake_ages <= upper_bound)
    )[0]
    if (SRS_df.loc[SRS_df['name'] == site, 'age lower bound']).item() > adjacent_sites_age_limit - 15:
        model += SRS_df.loc[SRS_df['name'] == site, 'displacement lower bound'] <= pl.lpSum(srs_displacements[n][index][i] * y[n][i] for n in constrained_event_indices for i in range(len(srs_displacements[n][0])))
    else:
        low_slip_rate = (SRS_df.loc[SRS_df['name'] == site, 'displacement lower bound'])/(SRS_df.loc[SRS_df['name'] == site, 'youngest upper bound'] - SRS_df.loc[SRS_df['name'] == site, 'oldest lower bound'])
        model += (low_slip_rate.item()*(upper_bound - lower_bound)) <= pl.lpSum(srs_displacements[n][index][i] * y[n][i] for n in constrained_event_indices for i in range(len(srs_displacements[n][0])))

#Create constraints for each SRS that was chosen to constrain (upper bound) with

for site in constrain_upper_bd_site_names:
    index = np.where(SRS_site_names == site)[0][0]
    position = SRS_df.loc[SRS_df['name'] == site, 'location']
    N_trench_index = np.searchsorted(trench_location, position) - 1
    S_trench_index = N_trench_index + 1
    if isinstance(N_trench_index, np.int64):
        adjacent_sites_age_limit = max(int(trench_oldest_ages[N_trench_index]), int(trench_oldest_ages[S_trench_index]))
    else:
        adjacent_sites_age_limit = max(int(trench_oldest_ages[N_trench_index[0]]), int(trench_oldest_ages[S_trench_index[0]]))        
    lower_bound = (SRS_df.loc[SRS_df['name'] == site, 'age lower bound']).item()
    #Set condition to make sure the events we are constraining happened when we have paleoseismic records on both sides
    if lower_bound <= adjacent_sites_age_limit:
        lower_bound = adjacent_sites_age_limit
    upper_bound = SRS_df.loc[SRS_df['name'] == site, 'age upper bound'].item()
    constrained_event_indices = np.where(
        (candidate_earthquake_ages >= lower_bound) & 
        (candidate_earthquake_ages <= upper_bound)
    )[0]
    if (SRS_df.loc[SRS_df['name'] == site, 'age lower bound']).item() > adjacent_sites_age_limit - 15:
        model += pl.lpSum(srs_displacements[n][index][i] * y[n][i] for n in constrained_event_indices for i in range(len(srs_displacements[n][0]))) <= SRS_df.loc[SRS_df['name'] == site, 'displacement upper bound']
    else:
        high_slip_rate = (SRS_df.loc[SRS_df['name'] == site, 'displacement upper bound'])/(SRS_df.loc[SRS_df['name'] == site, 'oldest upper bound'] - SRS_df.loc[SRS_df['name'] == site, 'youngest lower bound'])
        model += pl.lpSum(srs_displacements[n][index][i] * y[n][i] for n in constrained_event_indices for i in range(len(srs_displacements[n][0]))) <= (high_slip_rate.item()*(upper_bound - lower_bound))


#Optimize based on slip-per-event estimations
if SPE_optimize == 'on':
    if SPE_scenario == 'mean': #Optimize based on mean estimated slip per event
        for event in SPE_df['event']: 
            r = pl.LpVariable(f"SPE_r_{event}", lowBound=0) #Create residual variable for each event to optimize with
#Find which candidate events (which 'n') correspond to each trench event and find the displacements matrix that corresponds to the given site
            n_indices = np.where(candidates_df[event] == 1)[0]
            site = event.split('_')[0]
            index = trench_site_names.index(site)
#Create objective functions for all displacements associated with that event at that site
            model += r >= SPE_df.loc[SPE_df['event'] == event, 'mean'] - pl.lpSum(trench_displacements[n][index][i] * y[n][i] for n in n_indices for i in range(len(trench_displacements[n][0])))
            model += r >= -(SPE_df.loc[SPE_df['event'] == event, 'mean'] - pl.lpSum(trench_displacements[n][index][i] * y[n][i] for n in n_indices for i in range(len(trench_displacements[n][0]))))
            residual += r
    if SPE_scenario == 'max': #Optimize based on max estimated slip per event
        for event in SPE_df['event']: 
            r = pl.LpVariable(f"SPE_r_{event}", lowBound=0) #Create residual variable for each event to optimize with
#Find which candidate events (which 'n') correspond to each trench event and find the displacements matrix that corresponds to the given site
            n_indices = np.where(candidates_df[event] == 1)[0]
            site = event.split('_')[0]
            index = trench_site_names.index(site)
#Create objective functions for all displacements associated with that event at that site
            model += r >= SPE_df.loc[SPE_df['event'] == event, 'upper_bd'] - pl.lpSum(trench_displacements[n][index][i] * y[n][i] for n in n_indices for i in range(len(trench_displacements[n][0])))
            model += r >= -(SPE_df.loc[SPE_df['event'] == event, 'upper_bd'] - pl.lpSum(trench_displacements[n][index][i] * y[n][i] for n in n_indices for i in range(len(trench_displacements[n][0]))))
            residual += r
    if SPE_scenario == 'min': #Optimize based on min estimated slip per event
        for event in SPE_df['event']: 
            r = pl.LpVariable(f"SPE_r_{event}", lowBound=0) #Create residual variable for each event to optimize with
#Find which candidate events (which 'n') correspond to each trench event and find the displacements matrix that corresponds to the given site
            n_indices = np.where(candidates_df[event] == 1)[0]
            site = event.split('_')[0]
            index = trench_site_names.index(site)
#Create objective functions for all displacements associated with that event at that site
            model += r >= SPE_df.loc[SPE_df['event'] == event, 'lower_bd'] - pl.lpSum(trench_displacements[n][index][i] * y[n][i] for n in n_indices for i in range(len(trench_displacements[n][0])))
            model += r >= -(SPE_df.loc[SPE_df['event'] == event, 'lower_bd'] - pl.lpSum(trench_displacements[n][index][i] * y[n][i] for n in n_indices for i in range(len(trench_displacements[n][0]))))
            residual += r

#Optimize based on slip-per-event-group estimations
if SPE_optimize == 'on':
    if SPE_scenario == 'mean':
        for group in summed_d_events: 
                r = pl.LpVariable(f"SPE_r_{group}", lowBound=0) #Create residual variable for each event to optimize with
#Find which candidate events (which 'n') correspond to each trench event and find the displacements matrix that corresponds to the given site
                n_indices = []
#Go through each slip-per-event group and save the indices of the candidate events that each trench event belongs to
                for i in range(len(group)):
                    n_indices.append(np.where(candidates_df[group[i]] == 1)[0])
                n_indices = [item for sublist in n_indices for item in sublist]
                site = group[0].split('_')[0]
                index = trench_site_names.index(site)
#Create objective functions for all displacements associated with that group at that site
                model += r >= summed_d_mean - pl.lpSum(trench_displacements[n][index][i] * y[n][i] for n in n_indices for i in range(len(trench_displacements[n][0])))
                model += r >= -(summed_d_mean - pl.lpSum(trench_displacements[n][index][i] * y[n][i] for n in n_indices for i in range(len(trench_displacements[n][0]))))
                residual += r
    if SPE_scenario == 'min':
        for group in summed_d_events: 
                r = pl.LpVariable(f"SPE_r_{group}", lowBound=0) #Create residual variable for each event to optimize with
#Find which candidate events (which 'n') correspond to each trench event and find the displacements matrix that corresponds to the given site
                n_indices = []
#Go through each slip-per-event group and save the indices of the candidate events that each trench event belongs to
                for i in range(len(group)):
                    n_indices.append(np.where(candidates_df[group[i]] == 1)[0])
                n_indices = [item for sublist in n_indices for item in sublist]
                site = group[0].split('_')[0]
                index = trench_site_names.index(site)
#Create objective functions for all displacements associated with that group at that site
                model += r >= summed_d_lower_bd - pl.lpSum(trench_displacements[n][index][i] * y[n][i] for n in n_indices for i in range(len(trench_displacements[n][0])))
                model += r >= -(summed_d_lower_bd - pl.lpSum(trench_displacements[n][index][i] * y[n][i] for n in n_indices for i in range(len(trench_displacements[n][0]))))
                residual += r
    if SPE_scenario == 'max':
        for group in summed_d_events: 
                r = pl.LpVariable(f"SPE_r_{group}", lowBound=0) #Create residual variable for each event to optimize with
#Find which candidate events (which 'n') correspond to each trench event and find the displacements matrix that corresponds to the given site
                n_indices = []
#Go through each slip-per-event group and save the indices of the candidate events that each trench event belongs to
                for i in range(len(group)):
                    n_indices.append(np.where(candidates_df[group[i]] == 1)[0])
                n_indices = [item for sublist in n_indices for item in sublist]
                site = group[0].split('_')[0]
                index = trench_site_names.index(site)
#Create objective functions for all displacements associated with that group at that site
                model += r >= summed_d_upper_bd - pl.lpSum(trench_displacements[n][index][i] * y[n][i] for n in n_indices for i in range(len(trench_displacements[n][0])))
                model += r >= -(summed_d_upper_bd - pl.lpSum(trench_displacements[n][index][i] * y[n][i] for n in n_indices for i in range(len(trench_displacements[n][0]))))
                residual += r


#Constrain based on slip-per-event-group estimations
if SPE_constrain == 'on':
    for group in summed_d_events: 
#Find which candidate events (which 'n') correspond to each trench event and find the displacements matrix that corresponds to the given site
        n_indices = []
#Go through each slip-per-event group and save the indices of the candidate events that each trench event belongs to
        for i in range(len(group)):
            n_indices.append(np.where(candidates_df[group[i]] == 1)[0])
        n_indices = [item for sublist in n_indices for item in sublist]
        site = group[0].split('_')[0]
        index = trench_site_names.index(site)
#Create objective functions for all displacements associated with that group at that site
        model += summed_d_lower_bd <= pl.lpSum(trench_displacements[n][index][i] * y[n][i] for n in n_indices for i in range(len(trench_displacements[n][0]))) <= summed_d_upper_bd


#Create slip-per-event constraints
if SPE_constrain == 'on':
    for event in SPE_df['event']: 
#Find which candidate events (which 'n') correspond to each trench event and find the displacements matrix that corresponds to the given site
        n_indices = np.where(candidates_df[event] == 1)[0]
        site = event.split('_')[0]
        index = trench_site_names.index(site)
#Create objective functions for all displacement matrices associated with that site and those candidate events
        model += SPE_df.loc[SPE_df['event'] == event, 'lower_bd'].item() <= pl.lpSum(trench_displacements[n][index][i] * y[n][i] for n in n_indices for i in range(len(trench_displacements[n][0]))) <= SPE_df.loc[SPE_df['event'] == event, 'upper_bd'].item()

#Constrain slip using the default slip rate bounds for points of chosen spacing along fault
for location in default_slip_rate_locations:
    N_trench_index = np.searchsorted(trench_location, location) - 1
    S_trench_index = N_trench_index + 1
    if isinstance(N_trench_index, np.int64):
        adjacent_sites_age_limit = max(int(trench_oldest_ages[N_trench_index]), int(trench_oldest_ages[S_trench_index]))
    else:
        adjacent_sites_age_limit = max(int(trench_oldest_ages[N_trench_index[0]]), int(trench_oldest_ages[S_trench_index[0]]))
    constrained_event_indices = np.where(candidate_earthquake_ages >= adjacent_sites_age_limit)[0]
    index = np.where(default_slip_rate_locations == location)[0][0]
    #Put different default slip constraints on different sections of the fault
###IMPORTANT DECISION: to deal with problems resulting low slip rates at fault tips from the finite nature of the model, only constrain locations >20 km from the N end of the model (slip naturally tapers near San Gorgonio Pass)
    if 20 < location < 315:
        model += Big_Bend_default_high_rate*(1812 - adjacent_sites_age_limit) >= pl.lpSum(default_displacements[n][index][i] * y[n][i] for n in constrained_event_indices for i in range(len(default_displacements[n][0]))) >= Big_Bend_default_low_rate*((1812 - adjacent_sites_age_limit))
    if location > 315:
        model += San_Bernadino_default_high_rate*(1812 - adjacent_sites_age_limit) >= pl.lpSum(default_displacements[n][index][i] * y[n][i] for n in constrained_event_indices for i in range(len(default_displacements[n][0]))) >= San_Bernadino_default_low_rate*((1812 - adjacent_sites_age_limit))


#Constrain event size compared to other events at that same site (if SPE_relative_constrain == 'on')
if SPE_relative_constrain == 'on':
    for site_index in range(len(SPE_relative_list)):
        for event_idx in range(len(SPE_relative_list[site_index]) - 1):
            event = SPE_relative_list[site_index][event_index]
            next_event = SPE_relative_list[site_index][event_index + 1]
#Find which candidate events (which 'n') correspond to each trench event and find the displacements matrix that corresponds to the given site
            n_indices = np.where(candidates_df[event] == 1)[0] #Indices of the first event to be compared
            o_indices = np.where(candidates_df[next_event] == 1)[0] #Indices of the next event (larger event, according to arbitrary decision on how dataframe should be ordered) 
            site = event.split('_')[0]
            index = trench_site_names.index(site)
#Create objective functions for all displacement matrices associated with that site and those candidate events
            model += pl.lpSum(trench_displacements[n][index][i] * y[n][i] for n in n_indices for i in range(len(trench_displacements[n][0]))) <= pl.lpSum(trench_displacements[o][index][i] * y[o][i] for o in o_indices for i in range(len(trench_displacements[o][0])))


#Constrain event size to be comparable to other similar-sized events at that same site (within 'equality_threshold' meters, which is arbitrary measurement)
if SPE_relative_constrain == 'on':
    for group in equal_size_events:
        for idx in range(len(group) - 1): #Loop through each pair in the group of similar sized events to constrain displacement difference to be within equality_threshold
#Find which candidate events (which 'n') correspond to each trench event in the pair and find the displacements matrix that corresponds to the given site
            n_indices = [[], []]
#Go through each slip-per-event group and save the indices of the candidate events that each trench event belongs to
            n_indices[0].append(np.where(candidates_df[group[idx]] == 1)[0])
            n_indices[1].append(np.where(candidates_df[group[idx + 1]] == 1)[0])
        #n_indices = [item for sublist in n_indices for item in sublist]
            site = group[0].split('_')[0]
            index = trench_site_names.index(site)
#Create objective functions for all displacements associated with that group at that site
            model += - equality_threshold <= pl.lpSum(trench_displacements[n][index][i] * y[n][i] for n in n_indices[0][0] for i in range(len(trench_displacements[n][0]))) - pl.lpSum(trench_displacements[n][index][i] * y[n][i] for n in n_indices[1][0] for i in range(len(trench_displacements[n][0]))) <= equality_threshold



#Solve
model += residual


#Selection: choose wether to return single best-fitting history or family of best-fitting histories

Single_solution = False #Select single best event history vs. multiple simiarly-fitting histories
Solution_count = 10 #Choose how many optimal histories to keep (default = 10)

if Single_solution == True:
    solver = pl.GUROBI(
    msg=True,
    PoolSearchMode=0,
    MIPGap=0.01,
    TimeLimit=600,
    Symmetry=0
    )
    model.solve(solver)
if Single_solution == False:
    solver = pl.GUROBI(
    msg=True,
    PoolSearchMode=2,
    PoolSolutions=Solution_count,
    PoolGap=0.02,
    MIPGap=0.01,
    TimeLimit=600,
    Symmetry=0
    )
    model.solve(solver)
    gmodel = model.solverModel
    #solver.buildSolverModel(model)
    #gmodel = solver.solverModel
    gmodel.optimize()
    
#^Params:
# Report live updates from model while running
# 2 = search for 'n' best solutions, 1 = look for optimum and keep alternative solutions, 0 = only search for single optimum solution
# Keep up to 200 solutions from solutions that satisfy PoolGap constraint
# Keep solutions that are within this percentage (100 x val) difference from the optimal solution (BestBd): in other words, keep solutions with <= this 'Gap'
# Stop once solution falls below this percentage (100 x val) difference from the optimal solution (BestBd): in other words, stop once 'Gap' falls below this
# Stop model if solution is not found by this time limit (in s)
# How aggresively to break symmetry: in other words, how aggresively to cut branches in model with many similar solutions (from 0 = off to 2 = aggressive)

#Print off the candidate events of the best history that the model chooses
chosen_endpoints = []
for i in E:
    for n in y[i]:
        if pl.value(y[i][n]) == 1:
            chosen_endpoints.append(y[i][n])

#Print out chosen endpoint sets and minimized d (absolute deviation)
print("Chosen endpoints:", chosen_endpoints)
print("Selected total deviation:", pl.value(model.objective))

In [ ]:
from gurobipy import GRB

#Number of stored solutions
num_solutions = gmodel.SolCount

print(f"Number of solutions found: {num_solutions}")

#Create array to store selected variables for each solution
Solutions_array = [[] for _ in range(num_solutions)]

#Loop through all solutions
for i in range(num_solutions):

    #Select solution i
    gmodel.setParam(GRB.Param.SolutionNumber, i)

    #Objective value for solution i
    obj_val = gmodel.PoolObjVal

    print(f"\nSolution {i + 1} - Objective: {obj_val}")

    #Loop through variables in this solution
    for v in gmodel.getVars():

        #Store active binary variables
        if abs(v.Xn) > 0.0001:

            Solutions_array[i].append(v.VarName)

            print(v.VarName, v.Xn)

In [ ]:
#Plot results

s = 0 #KNOB: Plot first (best fitting) solution from family of solutions (change to plot different solutions from family of histories)

#Store the indices of the candidate events and their chosen endpoints encoded by decision variable 'y'

chosen = []
endpoint_index = []
for endpoint in Solutions_array[s][1:]:
    if endpoint[0] == 'y': #Pick out only the chosen decision variables, not the values of the residual variables
        numbers = re.findall(r'\d+', endpoint) #Pick out the integer that represents the chosen candidate event AND chosen endpoint set
        integer_list = [int(n) for n in numbers]
        chosen.append(integer_list[0])
        endpoint_index.append(integer_list[1])

#Store the endpoints chosen by model
Chosen_N_Rupture_Endpoints = []
Chosen_S_Rupture_Endpoints = []

for i in range(len(chosen)):
    Chosen_N_Rupture_Endpoints.append(N[chosen[i]][endpoint_index[i]])
    Chosen_S_Rupture_Endpoints.append(S[chosen[i]][endpoint_index[i]])

#Plot results as earthquake rupture strip map
event_times = []
for i in chosen:
    event_time = candidate_earthquake_ages[i]
    event_times.append(event_time)

trench_sum = np.zeros(len(candidate_events))
for i in range(len(candidate_events)):
    trench_sum[i] = site_binary.iloc[i].sum(axis = 0)

#Color lines based on how many trenches they ruptured:
chosen_colors = np.zeros(len(chosen), dtype = object)
for i in range(len(chosen)):
    if trench_sum[chosen[i]] == 8:
        chosen_colors[i] = 'yellow'
    elif trench_sum[chosen[i]] == 7:
        chosen_colors[i] = 'gold'
    elif trench_sum[chosen[i]] == 6:
        chosen_colors[i] = 'orange'
    elif trench_sum[chosen[i]] == 5:
        chosen_colors[i] = 'orangered'
    elif trench_sum[chosen[i]] == 4:
        chosen_colors[i] = 'darkorange'
    elif trench_sum[chosen[i]] == 3:
        chosen_colors[i] = 'red'
    elif trench_sum[chosen[i]] == 2:
        chosen_colors[i] = 'firebrick'
    else:
        chosen_colors[i] = 'darkred'

line_widths = []

#Give each line a thickness based on its calculated magnitude (exaggerated exponentially then scaled down for easier visualization)
for i in range(len(chosen)):
    x = 0.00004
    line_width = x*(4.5**((2*(np.log10(Chosen_S_Rupture_Endpoints[i] - Chosen_N_Rupture_Endpoints[i]) - c0 - c2*Fd))/(2*c1 - B2*c1) - (abs(np.log10(Wlim)) + B1)/(2 - B2)))
    line_widths.append(line_width)

for location in trench_location:
    plt.axvline(x=location, color='black', linestyle=':', linewidth=2, alpha=0.7)
plt.hlines(y=event_times, xmin=Chosen_N_Rupture_Endpoints, xmax=Chosen_S_Rupture_Endpoints, colors=chosen_colors, lw=line_widths)
plt.xlabel('Distance South From Highway 46 (km)')
plt.ylabel('Calendar Year')
plt.title('Southern San Andreas Rupture Strip Map')
plt.show()

In [ ]:
#For figures: include plot of 1812 and 1857 ruptures in black on plot with simulated history

#Plot results

s = 0 #KNOB: Plot first (best fitting) solution from family of solutions (change to plot different solutions from family of histories)

#Store the indices of the candidate events and their chosen endpoints encoded by decision variable 'y'

chosen = []
endpoint_index = []
for endpoint in Solutions_array[s][1:]:
    if endpoint[0] == 'y': #Pick out only the chosen decision variables, not the values of the residual variables
        numbers = re.findall(r'\d+', endpoint) #Pick out the integer that represents the chosen candidate event AND chosen endpoint set
        integer_list = [int(n) for n in numbers]
        chosen.append(integer_list[0])
        endpoint_index.append(integer_list[1])

#Store the endpoints chosen by model
Chosen_N_Rupture_Endpoints = []
Chosen_S_Rupture_Endpoints = []

for i in range(len(chosen)):
    Chosen_N_Rupture_Endpoints.append(N[chosen[i]][endpoint_index[i]])
    Chosen_S_Rupture_Endpoints.append(S[chosen[i]][endpoint_index[i]])

#Plot results as earthquake rupture strip map
event_times = []
for i in chosen:
    event_time = candidate_earthquake_ages[i]
    event_times.append(event_time)

trench_sum = np.zeros(len(candidates_df))
for i in range(len(candidates_df)):
    trench_sum[i] = site_binary.iloc[i].sum(axis = 0)

#Color lines based on how many trenches they ruptured:
chosen_colors = np.zeros(len(chosen), dtype = object)
for i in range(len(chosen)):
    if trench_sum[chosen[i]] == 8:
        chosen_colors[i] = 'yellow'
    elif trench_sum[chosen[i]] == 7:
        chosen_colors[i] = 'gold'
    elif trench_sum[chosen[i]] == 6:
        chosen_colors[i] = 'orange'
    elif trench_sum[chosen[i]] == 5:
        chosen_colors[i] = 'orangered'
    elif trench_sum[chosen[i]] == 4:
        chosen_colors[i] = 'red'
    elif trench_sum[chosen[i]] == 3:
        chosen_colors[i] = 'crimson'
    elif trench_sum[chosen[i]] == 2:
        chosen_colors[i] = 'firebrick'
    else:
        chosen_colors[i] = 'darkred'

line_widths = []

#Give each line a thickness based on its calculated magnitude (exaggerated exponentially then scaled down for easier visualization)
for i in range(len(chosen)):
    x = 0.00004
    line_width = x*(4.5**((2*(np.log10(Chosen_S_Rupture_Endpoints[i] - Chosen_N_Rupture_Endpoints[i]) - c0 - c2*Fd))/(2*c1 - B2*c1) - (abs(np.log10(Wlim)) + B1)/(2 - B2)))
    line_widths.append(line_width)

for location in trench_location:
    plt.axvline(x=location, color='black', linestyle=':', linewidth=2, alpha=0.7)
plt.hlines(y=event_times, xmin=Chosen_N_Rupture_Endpoints, xmax=Chosen_S_Rupture_Endpoints, colors=chosen_colors, lw=line_widths)
##New addition: plot 1812 and 1857 historic earthquakes
#Plot horizontal line to show end of simulated history at the year 1800
#plt.hlines(y=1800, xmin = 0, xmax = 386, colors='black', linestyle='-.', linewidth=1)
#1812 San Luis Capistrano Earthquake:
plt.hlines(y=1812, xmin=260, xmax=377, colors='black', linewidth=2) #Rupture extent certain
plt.hlines(y=1812, xmin=205, xmax=260, colors='black', linestyles='--', linewidth=2) #Rupture extent possible
plt.hlines(y=1812, xmin=377, xmax=386, colors='black', linestyles='--', linewidth=2) #Rupture extent possible
#1857 Fort Tejon Earthquake
plt.hlines(y=1857, xmin=0, xmax=303, colors='black', linewidth=3) #Rupture extent certain
##
plt.xlabel('Distance South From Highway 46 (km)')
plt.ylabel('Calendar Year')
plt.title('Southern San Andreas Rupture Strip Map')
plt.show()

In [ ]:
#Analyze results:

In [ ]:
#Plot trench site slip rates

chosen_trench_slip_rates = np.zeros(len(trench_location))
for n in range(len(trench_location)):
    constrained_chosen_indices = []
    for i in chosen:
        if candidate_earthquake_ages[i] > trench_oldest_ages[n]:
            constrained_chosen_indices.append(i)
    single_site_trench_displacements = sum(trench_displacements[i][n][endpoint_index[int(np.where(np.array(chosen) == i)[0])]] for i in constrained_chosen_indices)
    chosen_trench_slip_rates[n] = (1000*single_site_trench_displacements)/(1812 - trench_oldest_ages[n])

plt.bar(trench_location, chosen_trench_slip_rates, color='red', width = 7.0)
plt.xlabel("Distance South from Highway 46 (km)")
plt.ylabel("Slip rate (mm/yr)")
plt.title("Trench Site Slip Rates")
#Big Bend Default Slip Rates
plt.hlines(y = 1000*Big_Bend_default_low_rate, xmin = 0, xmax = 300, color = 'black', linestyle = "--", linewidth = 1, label = 'Low bound slip rate')
plt.hlines(y = 1000*Big_Bend_default_high_rate, xmin = 0, xmax = 300, color = 'black', linestyle = "--", linewidth = 1, label = 'High bound slip rate')
#San Bernardino Default Slip Rates
plt.hlines(y = 1000*San_Bernadino_default_low_rate, xmin = 300, xmax = 386, color = 'black', linestyle = "--", linewidth = 1, label = 'High bound slip rate')
plt.hlines(y = 1000*San_Bernadino_default_high_rate, xmin = 300, xmax = 386, color = 'black', linestyle = "--", linewidth = 1, label = 'High bound slip rate')

#plt.axhline(y = 1000*((default_high_rate + default_low_rate)/2), color = 'red', linestyle = "dotted", linewidth = 1, label = 'Mean slip rate')

In [ ]:
#Plot slip rate site slip rates

chosen_srs_slip_rates = np.zeros(len(SRS_site_names))
for n in range(len(SRS_site_names)):
    constrained_chosen_indices = []
    ##
    position = SRS_df.loc[SRS_df['name'] == SRS_site_names[n], 'location'].item()
    N_trench_index = np.searchsorted(trench_location, position) - 1
    S_trench_index = N_trench_index + 1
    if isinstance(N_trench_index, np.int64):
        adjacent_sites_age_limit = max(int(trench_oldest_ages[N_trench_index]), int(trench_oldest_ages[S_trench_index]))
    else:
        adjacent_sites_age_limit = max(int(trench_oldest_ages[N_trench_index[0]]), int(trench_oldest_ages[S_trench_index[0]]))        
    lower_bound = (SRS_df.loc[SRS_df['name'] == SRS_site_names[n], 'age lower bound']).item()
    #Set condition to make sure the events we are constraining happened when we have paleoseismic records on both sides
    if lower_bound <= adjacent_sites_age_limit:
        lower_bound = adjacent_sites_age_limit
    ##
    for i in chosen:
        if SRS_df.loc[SRS_df['name'] == SRS_site_names[n], 'age upper bound'].item() >= candidate_earthquake_ages[i] > lower_bound:
            constrained_chosen_indices.append(i)
    single_site_srs_displacements = sum(srs_displacements[i][n][endpoint_index[int(np.where(np.array(chosen) == i)[0])]] for i in constrained_chosen_indices)
    chosen_srs_slip_rates[n] = (1000*single_site_srs_displacements)/(1812 - lower_bound)

plt.bar(np.array(SRS_df['location']), chosen_srs_slip_rates, color='green', width = 5.0)
plt.xlabel("Distance South from Highway 46 (km)")
plt.ylabel("Slip rate (mm/yr)")
plt.title("Slip Rate Site Slip Rates")
#Big Bend Default Slip Rates
plt.hlines(y = 1000*Big_Bend_default_low_rate, xmin = 0, xmax = 300, color = 'black', linestyle = "--", linewidth = 1, label = 'Low bound slip rate')
plt.hlines(y = 1000*Big_Bend_default_high_rate, xmin = 0, xmax = 300, color = 'black', linestyle = "--", linewidth = 1, label = 'High bound slip rate')
#San Bernardino Default Slip Rates
plt.hlines(y = 1000*San_Bernadino_default_low_rate, xmin = 300, xmax = 386, color = 'black', linestyle = "--", linewidth = 1, label = 'High bound slip rate')
plt.hlines(y = 1000*San_Bernadino_default_high_rate, xmin = 300, xmax = 386, color = 'black', linestyle = "--", linewidth = 1, label = 'High bound slip rate')

#plt.axhline(y = 1000*((default_high_rate + default_low_rate)/2), color = 'red', linestyle = "dotted", linewidth = 1, label = 'Mean slip rate')

In [ ]:
#Make bar graph of slip rates at default slip rate locations

slip_rates = []

for location in default_slip_rate_locations:
    N_trench_index = np.searchsorted(trench_location, location) - 1
    S_trench_index = N_trench_index + 1
    if isinstance(N_trench_index, np.int64):
        adjacent_sites_age_limit = max(int(trench_oldest_ages[N_trench_index]), int(trench_oldest_ages[S_trench_index]))
    else:
        adjacent_sites_age_limit = max(int(trench_oldest_ages[N_trench_index[0]]), int(trench_oldest_ages[S_trench_index[0]]))
    constrained_chosen_indices = []
    for n in chosen:
        if candidate_earthquake_ages[n] > adjacent_sites_age_limit:
            constrained_chosen_indices.append(n)
    constrained_endpoint_indices = []
    indices = np.where(np.array(event_times) > adjacent_sites_age_limit)[0]
    for i in indices:
        constrained_endpoint_indices.append(endpoint_index[i])
    index = np.where(default_slip_rate_locations == location)[0][0]
    displacements = []
    for i in range(len(indices)):
        displacements.append(default_displacements[constrained_chosen_indices[i]][index][constrained_endpoint_indices[i]])
    slip_rate = (sum(displacements))/(1812-adjacent_sites_age_limit)
    #Convert to mm/yr
    slip_rate = 1000*slip_rate[0]
    slip_rates.append(slip_rate)

plt.bar(np.arange(trench_location[0],trench_location[-1],default_slip_rate_constraint_spacing), slip_rates, color='blue', width = 3.0)
plt.xlabel("Distance South from Highway 46 (km)")
plt.ylabel("Slip rate (mm/yr)")
plt.title("Slip Rate Constraint Slip Rates")
#Big Bend Default Slip Rates
plt.hlines(y = 1000*Big_Bend_default_low_rate, xmin = 0, xmax = 300, color = 'black', linestyle = "--", linewidth = 1, label = 'Low bound slip rate')
plt.hlines(y = 1000*Big_Bend_default_high_rate, xmin = 0, xmax = 300, color = 'black', linestyle = "--", linewidth = 1, label = 'High bound slip rate')
#San Bernardino Default Slip Rates
plt.hlines(y = 1000*San_Bernadino_default_low_rate, xmin = 300, xmax = 386, color = 'black', linestyle = "--", linewidth = 1, label = 'High bound slip rate')
plt.hlines(y = 1000*San_Bernadino_default_high_rate, xmin = 300, xmax = 386, color = 'black', linestyle = "--", linewidth = 1, label = 'High bound slip rate')

#plt.axhline(y = 1000*((default_high_rate + default_low_rate)/2), color = 'red', linestyle = "dotted", linewidth = 1, label = 'Mean slip rate')

In [ ]:
#V2: Make bar graph of slip rates at default slip rate locations,
#but assume catalog is complete (all slip rates calculated from the oldest age that the record goes back to)

slip_rates = []

for location in default_slip_rate_locations:
    N_trench_index = np.searchsorted(trench_location, location) - 1
    S_trench_index = N_trench_index + 1
    if isinstance(N_trench_index, np.int64):
        adjacent_sites_age_limit = max(int(trench_oldest_ages[N_trench_index]), int(trench_oldest_ages[S_trench_index]))
    else:
        adjacent_sites_age_limit = max(int(trench_oldest_ages[N_trench_index[0]]), int(trench_oldest_ages[S_trench_index[0]]))
    constrained_chosen_indices = []
    for n in chosen:
        if candidate_earthquake_ages[n] > adjacent_sites_age_limit:
            constrained_chosen_indices.append(n)
    constrained_endpoint_indices = []
    indices = np.where(np.array(event_times) > adjacent_sites_age_limit)[0]
    for i in indices:
        constrained_endpoint_indices.append(endpoint_index[i])
    index = np.where(default_slip_rate_locations == location)[0][0]
    displacements = []
    for i in range(len(indices)):
        displacements.append(default_displacements[constrained_chosen_indices[i]][index][constrained_endpoint_indices[i]])
    slip_rate = (sum(displacements))/(1812-487)
    #Convert to mm/yr
    slip_rate = 1000*slip_rate[0]
    slip_rates.append(slip_rate)

plt.bar(np.arange(trench_location[0],trench_location[-1],default_slip_rate_constraint_spacing), slip_rates, color='blue', width = 3.0)
plt.xlabel("Distance South from Highway 46 (km)")
plt.ylabel("Slip rate (mm/yr)")
plt.title("Slip Rate Constraint Slip Rates")
#Big Bend Default Slip Rates
plt.hlines(y = 1000*Big_Bend_default_low_rate, xmin = 0, xmax = 300, color = 'black', linestyle = "--", linewidth = 1, label = 'Low bound slip rate')
plt.hlines(y = 1000*Big_Bend_default_high_rate, xmin = 0, xmax = 300, color = 'black', linestyle = "--", linewidth = 1, label = 'High bound slip rate')
#San Bernardino Default Slip Rates
plt.hlines(y = 1000*San_Bernadino_default_low_rate, xmin = 300, xmax = 386, color = 'black', linestyle = "--", linewidth = 1, label = 'High bound slip rate')
plt.hlines(y = 1000*San_Bernadino_default_high_rate, xmin = 300, xmax = 386, color = 'black', linestyle = "--", linewidth = 1, label = 'High bound slip rate')

#plt.axhline(y = 1000*((default_high_rate + default_low_rate)/2), color = 'red', linestyle = "dotted", linewidth = 1, label = 'Mean slip rate')

In [ ]:
#Make bar graph plotting slip-per-event residuals as a percentage of the range of acceptable values

#Store all SPE residuals from ILP Model
#Save the residuals and names of each slip per event residual variable in their own array

SPE_residuals = []
names = []

for v in gmodel.getVars():
        #Store SPE Residual variables
    if v.VarName[2] == 'E':
        name = v.VarName
        name = name.split('E_r_')[1]
        names.append(name)
        SPE_residuals.append(v.Xn)

SPE_residual_array = []
#Create new names array to deal with problem of extra name saved above for joint residual
SPE_names = []

for i in range(len(SPE_df['event'])):
    residual = SPE_residuals[i]/(SPE_df.loc[SPE_df['event'] == names[i], 'upper_bd'].iloc[0] - SPE_df.loc[SPE_df['event'] == names[i], 'lower_bd'].iloc[0])
    SPE_residual_array.append(residual)
    SPE_names.append(names[i])
    
#Make plot wider
plt.figure(figsize=(10, 3))
plt.bar(SPE_names, SPE_residual_array, color='orange', width = 0.5)
#Rotate labels
plt.xticks(rotation=90, ha='right')
plt.xlabel("Event Name")
plt.ylabel("Residual/Displacement Range")
plt.title("Proportional Slip Per Event Residuals")

In [ ]:
#Make bar graph plotting slip rate residuals as a percentage of the range of acceptable values

#Store all SRS residuals from ILP Model
#Save the residuals and names of each slip rate site residual variable in their own array

SRS_residuals = []
names = []

for v in gmodel.getVars():
        #Store SRS Residual variables
    if v.VarName[2] == 'S':
        name = v.VarName
        name = name.split('S_r_')[1]
        name = name.replace("_", " ")
        names.append(name)
        SRS_residuals.append(v.Xn)

SRS_residual_array = []

for i in range(len(optimize_site_names)):
    residual = SRS_residuals[i]/(SRS_df.loc[SRS_df['name'] == names[i], 'displacement upper bound'] - SRS_df.loc[SRS_df['name'] == names[i], 'displacement lower bound'])
    residual = residual.item()
    SRS_residual_array.append(residual)
    
#Make plot wider
plt.figure(figsize=(9, 3))
plt.bar(names, SRS_residual_array, color='orange', width = 1)
#Rotate labels
plt.xticks(rotation=45, ha='right')
plt.xlabel("Slip Rate Site Name")
plt.ylabel("Residual/Displacement Range (m)")
plt.title("Proportional Slip Rate Site Residuals")

In [ ]:
#Plot magnitude distribution of generated history

magnitudes_array = np.arange(6.5, 8, 0.1)
chosen_magnitudes = []
for i in range(len(chosen)):
    rupture_l = Chosen_S_Rupture_Endpoints[i] - Chosen_N_Rupture_Endpoints[i]
    index = (np.abs(possible_rupture_length/1000 - (rupture_l))).argmin()
    magnitude = (index)*0.01 + 6.5
    chosen_magnitudes.append(magnitude)

magnitude_sum = []

#Go through each precalculated magnitude, count the number of events that are closest to that magnitude, and append to a list that will be plotted on a bar graph
for magnitude in magnitudes_array:
    indices = []
    for i in chosen_magnitudes:
        if abs(i - magnitude) < 0.05:
            indices.append(i)
    magnitude_sum.append(len(indices))

plt.bar(magnitudes_array, magnitude_sum, color = 'black', width = 0.09)
plt.xlabel("Magnitude")
plt.ylabel("Count")
plt.title("Magnitude Distribution of Generated History")

In [ ]:
##END OF CURRENT WORK##

In [ ]:
#IDEA TO TEST: fit interevent times along the whole of the San Andreas (and then smaller segments of the San Andreas,
#i.e. a large stretch bounded by structural compleixites) to different hazard functions - is it Poissonian, or is there some
#self-organization within the larger system?

In [ ]:
#IDEA TO TEST: look at interevent times at regularly spaced points along fault, to analyze the effects of structural complexity/passing probabilities
#on hazard at different points along the fault

In [ ]:
#Read Kate Scharer's Maximum rupture model paper again: is the reason that one possible full length rupture is split into
#2 smaller ruptures is because of a regional climatic event used to differentiate the two events in the trenches?

In [ ]:
#Since we're calculating hazard functions for points that are continuous along fault, is there any way to integrate
#these hazard functions along strike...? (Not quite sure what that would do and what the purpose would be but sounds cool)

In [ ]:
#Plan: have two different interevent times versions: one where we generate 10,000 random samples from the ages above a
#certain probability threshold (which can be modified), and one where we generate 10,000 random samples from the entire age range

In [ ]:
#Interevent Times Code

#Create all possible interevent time combinations for the best fitting history (RI Ensemble) and compare to different distributions, then repeat 
#for arbitrary (1,000? 10,000?) number of best-fitting solutions

from scipy.stats import weibull_min
from scipy.stats import expon

#Make list of the names of every candidate_earthquake created in Event Enumeration
candidate_earthquake_names = []
for key in candidate_earthquakes.keys():
    candidate_earthquake_names.append(key)

#Put indices from chosen in chronological order, then use to pull out their corresponding earthquake objects from candidate_earthquakes by indexing 'candidate_earthquake_names' list
#Randomly sample from these ages until 10,000 combinations at one year resolution have been generated
#Calculate log likelihood for each distribution for each combination, store in array (to be averaged when making conclusions)

#Distributions to be tested: Weibull, BPT, Exponential (Poisson)

#Remember stratigraphic order constraint and 10 year minimum interval constraint (supported by trenching)

# >> REMOVE THIS LINE WHEN CODE IS RUN COMPLETELY: 'chosen' here is for testing purposes only

#chosen = [4, 34, 77, 92, 256, 650, 800, 900]

# >>

n = 10000 #Number of interevent sets to be sampled and tested against different distributions
probability_option = 'high' #Options: 'default': pull n samples from all ages in each earthquake object, 'high':  pull n samples from ages above a certain threshold (set below)
sampling_threshold = 10**-4
sampling_option = 'interval' #Options: 'manual': calculate interevent statistics from specified point(s) along fault, 'interval': calculate interevent statistics from points of constant spacing along fault
intervals = 100 #(If sampling_option = 'interval') Distance (in kilometers) between sampled points
points = [70, 250, 350] #(If sampling_option = 'manual') Coordinates (in terms of distance along strike (in kilometers)) of sampled points

ages = [[] for _ in range(len(points))]

#From 'chosen', pull the indices of the events that ruptured given point along fault

if sampling_option == 'manual':
    sampled_chosen = [[] for _ in range(len(points))]

if sampling_option == 'interval':
    points = np.arange(0, fault_length, intervals)
    sampled_chosen = [[] for _ in range(len(points))]

#Store the indices of the events that ruptured through a given point
for i in range(len(points)):
    for j in range(len(chosen)):
        if Chosen_N_Rupture_Endpoints[j] < points[i] < Chosen_S_Rupture_Endpoints[j]:
            sampled_chosen[i].append(chosen[j])

#Loop entire interevent times workflow through each point chosen for analysis

for z in range(len(sampled_chosen)):

    for idx in sampled_chosen[z]:
        ages[z].append(candidate_earthquake_ages[idx])

    ageidx = zip(ages[z], sampled_chosen[z])
    ageidx = list(sorted(ageidx))

    sortedselected = []

    for i in range(len(ageidx)):
        sortedselected.append(ageidx[i][1])

    age_array = [[] for _ in range(len(sampled_chosen[z]))]

##NOTE: This is for all San Andreas events combined, to test for large-scale patterns. If there are no fault-scale interactions, distribution should be best fit by Exponential distribution
##To create version for points evenly spaced along fault, find the events that correspond to the endpoints for which the desired point is inbetween and only use these events

#Pull 10,000 combinations of ages

    if probability_option == 'default':
    
        for i in range(len(sampled_chosen[z])):
            age_array[i].append(candidate_earthquakes[candidate_earthquake_names[sortedselected[i]]].age)

    ##Pull 10,000 combinations of ages above a certain probability threshold

    if probability_option == 'high':

        for i in range(len(sampled_chosen[z])):
            indices = np.where(candidate_earthquakes[candidate_earthquake_names[sortedselected[i]]].PDF > sampling_threshold)[0]
            age_array[i].append(candidate_earthquakes[candidate_earthquake_names[sortedselected[i]]].age[indices])

        #Pull 10,000 random interevent time arrays from age_array and store in a different array, to be used in next step
    event_times = [[] for _ in range(n)]

    
    for i in range(n):
        while True:
            event_set = []
            for j in range(len(sampled_chosen[z])):
                event_set.append(random.sample(age_array[j][0].tolist(), k = 1))

        ## Stratigraphic order and 10-year minimum interevent time constraint: if violated, restart sampling of ith event set
            stratigraphic_order = True
        
            for x in range(len(sampled_chosen[z]) - 1): #Make sure events are still in stratigraphic order and interevent time > 10 years
                if event_set[x + 1][0] - event_set[x][0] < 10:
                    stratigraphic_order = False
                    break

            if stratigraphic_order:
                event_times[i] = event_set
                break

#Calculate interevent times

    interevent_times = [[] for _ in range(n)]

    for i in range(n):
        for j in range(len(sampled_chosen[z]) - 1):
            interevent_times[i].append(event_times[i][j + 1][0] - event_times[i][j][0])

    #Put interevent times in order (shortest to longest)

    for i in range(n):
        interevent_times[i] = np.array(sorted(interevent_times[i])) #Convert to an array to allow easier computations in next cell

#Fit each interevent time set to each of three distributions (Poisson, Brownian Passage Time, Weibull), calculate each's log likelihood, and plot as histogram



#Initialize arrays that will hold log probability of each distribution for each history

    BPT_probabilities = np.zeros(n)
    Weibull_probabilities = np.zeros(n)
    Exponential_probabilities = np.zeros(n)

    for i in range(n):
#Calculate parameters for each distribution

    #Exponential (and others)
        mu = np.mean(interevent_times[i])

    #Brownian Passage Time
        alpha = np.sqrt(mu/len(interevent_times[i]) * sum(1/interevent_times[i] - 1/mu))

    #Weibull
        shape, loc, scale = weibull_min.fit(interevent_times[i], floc = 0)

#Calculate log likelihood of the history for all distributions  --> Store in their respective arrays

    #Brownian Passage Time
        BPT_prob = sum(np.log(np.sqrt(mu/(2*np.pi * alpha**2 * interevent_times[i]**3)) * np.e**(-(((interevent_times[i] - mu)**2)/(2 * mu * alpha**2 * interevent_times[i])))))
        BPT_probabilities[i] = BPT_prob
    
    #Weibull
        Weibull_prob = sum(np.log(weibull_min.pdf(interevent_times[i], shape, scale = scale)))
        Weibull_probabilities[i] = Weibull_prob

    #Exponential
        Exponential_prob = sum(np.log(expon.pdf(interevent_times[i], scale = mu)))
        Exponential_probabilities[i] = Exponential_prob

    #Create large plot showing results for every point sampled along the fault

fig, axes = plt.subplots(1, len(points), figsize = (20,4))

for i in range(len(points)):
    axes[i].hist([BPT_probabilities, Weibull_probabilities, Exponential_probabilities], bins = 'auto', color = ['orange', 'blue', 'black'])
    axes[i].set_title("Hazard Functions Fit to Earthquake Recurrence", points[i], "km south of Highway 46")
        
plt.tight_layout
plt.show()

In [ ]:
chosen

In [ ]:
# ^^ Current condensed, final draft code to plot results for all sampled points along fault

In [ ]:
#Fit each interevent time set to each of three distributions (Poisson, Brownian Passage Time, Weibull), calculate each's log likelihood, and plot as histogram

from scipy.stats import weibull_min
from scipy.stats import expon

#Initialize arrays that will hold log probability of each distribution for each history

BPT_probabilities = np.zeros(n)
Weibull_probabilities = np.zeros(n)
Exponential_probabilities = np.zeros(n)

for i in range(n):
#Calculate parameters for each distribution

    #Exponential (and others)
    mu = np.mean(interevent_times[i])

    #Brownian Passage Time
    alpha = np.sqrt(mu/len(interevent_times[i]) * sum(1/interevent_times[i] - 1/mu))

    #Weibull
    shape, loc, scale = weibull_min.fit(interevent_times[i], floc = 0)

#Calculate log likelihood of the history for all distributions  --> Store in their respective arrays

    #Brownian Passage Time
    BPT_prob = sum(np.log(np.sqrt(mu/(2*np.pi * alpha**2 * interevent_times[i]**3)) * np.e**(-(((interevent_times[i] - mu)**2)/(2 * mu * alpha**2 * interevent_times[i])))))
    BPT_probabilities[i] = BPT_prob
    
    #Weibull
    Weibull_prob = sum(np.log(weibull_min.pdf(interevent_times[i], shape, scale = scale)))
    Weibull_probabilities[i] = Weibull_prob

    #Exponential
    Exponential_prob = sum(np.log(expon.pdf(interevent_times[i], scale = mu)))
    Exponential_probabilities[i] = Exponential_prob

In [ ]:
import matplotlib.pyplot as plt

#Plot histograms of log likelihoods
plt.hist([BPT_probabilities, Weibull_probabilities, Exponential_probabilities], bins = 'auto', color = ['orange', 'blue', 'black'])

In [ ]:
##

In [ ]:
#Abandoned 'Event Quality' code

#Insert to Event Enumeration (remove events based on 'event quality', per Scharer)

event_quality = {
    'trench': [],
    'event name': [],
    'quality score': []
}

Event_Quality_df = pd.DataFrame(event_quality)

## (REMOVE LATER)
drop_indices = []
event_quality_threshold = 2
##

for i in range(len(Event_Quality_df['quality score'])):
    if Event_Quality_df['quality score'].iloc[i] < event_quality_threshold:
        site = Event_Quality_df['trench'][i]
        name = Event_Quality_df['event name'][i]
        index = events.index[(events['site'] == site) & (events['event'] == name)]
        drop_indices.append(index[0])

In [ ]:
##Old deprecated event enumeration code (overloads for larger trench datasets)

import numpy as np
import pandas as pd
from itertools import product
from scipy.interpolate import interp1d
import itertools
from itertools import product

Enumeration_Option = 2 #Option 1 = PDF Enumeration, Option 2 = CDF Enumeration
threshold = 0.0003 #Change to set how well event pdfs need to overlap to be included in candidates (default = 0.0003)

#Define sites dataframe for Southern San Andreas Fault
sites = pd.DataFrame({
##Test: Remove new sites, run with old sites to see what is making model infeasible    "site": ["end_n", "Annette", "Bidart", "Frazier", "Elizabeth", "Pallett", "Wrightwood", "Pitman", "Burro", "end_s"],
    "site": ["end_n", "Bidart", "Frazier", "Elizabeth", "Pallett", "Wrightwood", "end_s"],
    #"site": ["end_n", "Annette", "Bidart", "Frazier", "Elizabeth", "Pallett", "Wrightwood", "Pitman", "Burro", "end_s"],
    #"loc": [0, 20, 72, 169, 218, 270, 292, 317, 377, 386],
    "loc": [0, 72, 169, 218, 270, 292, 386],
    #"file": [None, "C:/Users/erikc/OneDrive/Documents/Research/Annette_inverted.csv", "C:/Users/erikc/OneDrive/Documents/Research/Bidart.csv", "C:/Users/erikc/OneDrive/Documents/Research/Frazier.csv", "C:/Users/erikc/OneDrive/Documents/Research/Elizabeth_Lake_test.csv", "C:/Users/erikc/OneDrive/Documents/Research/PCreek.csv", "C:/Users/erikc/OneDrive/Documents/Research/Wright.csv", "C:/Users/erikc/OneDrive/Documents/Research/PitmanCanyon.csv", "C:/Users/erikc/OneDrive/Documents/Research/BurroFlats.csv",  None],
    "file": [None, "C:/Users/erikc/OneDrive/Documents/Research/Bidart.csv", "C:/Users/erikc/OneDrive/Documents/Research/Frazier.csv", "C:/Users/erikc/OneDrive/Documents/Research/Elizabeth_Lake_test.csv", "C:/Users/erikc/OneDrive/Documents/Research/PCreek.csv", "C:/Users/erikc/OneDrive/Documents/Research/Wright.csv",  None],
    #"event_code": [None, "Event", "Event", "Event", "Event", "Event", "Event", "Event", "Event", None],
    "event_code": [None, "Event", "Event", "Event", "Event", "Event", None],
    #"earliest": [None, 893, 1350, 601, 1090, 630, 420, 816, -56, None]
    "earliest": [None, 1350, 601, 1090, 630, 420, None]
})


if Enumeration_Option == 1:
        #Create earthquake class
    class Earthquake:

        #Constructor: stores earthquake name, age, PDF, and CDF for each earthquake event (i.e. Bidart.Event_2)
        def __init__(self, name, age, PDF, CDF):
            self.name = name
            self.age = np.array(age)
            self.PDF = np.array(PDF)
            self.CDF = np.array(CDF)
        #Returns single sampled age based on PDF: generate random number between 0 and 1, then sample the age from the CDF that corresponds to number
        def event(self):
            r = np.random.rand()
            return self.age[np.argmin(np.abs(self.CDF - r))]
        #Returns n sampled ages: calls event() n times, then stores results in an array
        def events(self, n):
            return np.array([self.event() for _ in range(n)])
        #Resamples earthquake age to yearly resolution: calculates age range (L), then creates list of years inside age range and interpolates event PDF and CDF to those years
        def resample(self):
            L = int(self.age[-1] - self.age[0])
            new_age = np.linspace(self.age[0], self.age[-1], L + 1)

            f_age = interp1d(self.age, self.age, fill_value="extrapolate")
            f_pdf = interp1d(self.age, self.PDF, fill_value="extrapolate")
            f_cdf = interp1d(self.age, self.CDF, fill_value="extrapolate")

            self.age = f_age(new_age)
            self.PDF = f_pdf(new_age)
            self.PDF = self.PDF / np.sum(self.PDF)
            self.CDF = f_cdf(new_age)
        #Calculates joint probability of multiple earthquake events: find age spacing, and make sure both events have same spacing
        def jointprobability(self, other):
            diff_age = np.mean(np.diff(self.age))
            if diff_age != np.mean(np.diff(other.age)):
                raise ValueError("Sampling interval differs")
            #Calculate misallignment between both age groups, then use this to shift the two distributions so they line up
            misalign = (other.age[0] % diff_age) - (self.age[0] % diff_age)
            self_age = self.age.copy()
            other_age = other.age.copy()
            self_age = self_age + misalign
            #Find where distributions overlap
            lower = max(min(self_age), min(other_age))
            upper = min(max(self_age), max(other_age))

            if lower >= upper:
                return None
            #Create grid of shared ages
            new_age = np.arange(lower, upper + diff_age, diff_age)
            #Find starting and ending indexes for both event PDFs
            this_min = np.argmin(np.abs(self_age - lower))
            that_min = np.argmin(np.abs(other_age - lower))
            this_max = np.argmin(np.abs(self_age - upper))
            that_max = np.argmin(np.abs(other_age - upper))
            #Multiply PDFs element-wise
            new_pdf = self.PDF[this_min:this_max+1] * other.PDF[that_min:that_max+1]
            #Normalize (set sum of new PDF to 1)
            overlap = np.sum(new_pdf)
            new_pdf = new_pdf / overlap
            new_cdf = np.cumsum(new_pdf)

            return Earthquake(self.name, new_age, new_pdf, new_cdf)


    #Define function load_oxcal: Reads in events from each Oxcal file, pulls out and saves ages, pdf, and cdf data (outputs 'Earthquake' object)
    def load_oxcal(filename, name_selector='Event', delim=None):
        if delim:
            df = pd.read_csv(filename, sep=delim)
        else:
            df = pd.read_csv(filename)
    
        # Keep only posterior type
        df = df[df["type"] == "posterior"]
        df = df[df["name"].str.contains(name_selector)]

        result = []

        for eventname, group in df.groupby("name"):
            ages = group["value"].values
            pdf = group["probability"].values
            cdf = np.cumsum(pdf)
            sumcdf = cdf[-1]

            result.append(
                Earthquake(
                    name=eventname,
                    age=ages,
                    PDF=pdf / sumcdf,
                    CDF=cdf / sumcdf
                )
            )
        return result


        for eventname in events["name"].unique():
            event = events[events["name"] == eventname]


            ages = event["value"].values
            pdf = event["probability"].values
            cdf = np.cumsum(pdf)

            sumcdf = cdf[-1]
            result.append(
                Earthquake(
                    name=eventname,
                    age=ages,
                    PDF=pdf / sumcdf,
                    CDF=cdf / sumcdf
                )
            )
        return result


    #Define function get_marginal_pdfs: Builds dataframe with age, PDF, and name information as columns in dataframe with events as rows, 
    #also only keeps nonzero rows, and gives each event its own number
    def get_marginal_pdfs(oxcal):
        rows = []
        for i, eq in enumerate(oxcal):
            df = pd.DataFrame({
                "age": eq.age,
                "PDF": eq.PDF,
                "name": eq.name
            })
            df = df[df["age"] > 0]
            df["site_order"] = i
            rows.append(df)
        return rows

    #Define function as_binary_matrix: loops over dataframe to convert each row into binary
    def as_binary_matrix(n, width):
        return np.array([
            list(np.binary_repr(i, width=width))
            for i in range(1, n + 1)
        ], dtype=int)
##
#Original location for 'define sites dataframe'
##
    #Create marginal PDF dataframes for each paleoseismic site using load_oxcal function
    marginal_data = []

    for i, row in sites.iterrows():
        if pd.isna(row["file"]):
            df = pd.DataFrame({"age": [np.nan], "PDF": [np.nan], "name": [np.nan], "CDF": [np.nan]})
        else:
            oxcal = load_oxcal(row["file"], row["event_code"])
            df = pd.concat(get_marginal_pdfs(oxcal), ignore_index=True)

        df["site"] = row["site"]
        marginal_data.append(df)

    sites["marginal_pdf"] = marginal_data

    #Convert marginal_pdf to list, dropping column and row names
    marginal = pd.concat(sites["marginal_pdf"].tolist(), ignore_index=True)
    marginal = marginal.dropna(subset=["name"])

    #Converts data type to float, exchanges 'age' and 'PDF' index with 'name' and 'event'
    marginal["year"] = marginal["age"].astype(float)
    marginal["prob"] = marginal["PDF"].astype(float)
    marginal = marginal.drop(columns=["age", "PDF"])
    marginal = marginal.rename(columns={"name": "event"}) 

    #Format each trench event to be labeled as site.event_# (i.e. Bidart.Event_2)
    marginal["event"] = marginal["event"].str.replace(r"\s+", "_", regex=True)
    marginal["event_name"] = marginal["site"] + "_" + marginal["event"]

    print("Marginal dataframe constructed...")


##Added sort=False because if groupby sorts alphabetically, indexing for jointprobability will be screwed up
    #Calculate mean year for each event (weighted by probability)
    events = (
        marginal.groupby(["site", "event"], sort=False)
        .apply(lambda x: np.average(x["year"], weights=x["prob"]))
        .reset_index(name="year")
    )
    n_events = len(events)

    ##Moved drop historical events code here and edited to work on events
    #Removes historical events
    historical = marginal[marginal["prob"] == 1]["event_name"].unique()
    #Specific to San Andreas Fault: Remove EL1 and Event_X (known to be 1857 and 1812 historical events, respectively)
    historical = np.append(historical, 'Elizabeth_Event_EL1')
    historical = np.append(historical, 'Pallett_Event_X')

    events_drop_indices = []
    for event in historical:
        events_drop_indices.append(events[(events['site'] == event.split('_')[0]) & (events['event'] == event.split('_', 1)[-1])].index[0])
    #Specific to San Andreas fault: Drop index = 5, which is the historical Annette groundshaking event
    events_drop_indices.append(5)
    events.drop(events.index[events_drop_indices], inplace = True)

    #Reset indices after dropping historical events
    events.reset_index(drop=True, inplace=True)

    #To be used later: event count after deletion of historical events
    n_events = len(events)

##

    #Build candidates matrix, using similar formatting from above to label event_names, then initiliazing candidates as an identity matrix
    event_names = (events["site"] + "_" + events["event"]).values
    candidates = np.zeros((n_events, n_events), dtype=int)
    np.fill_diagonal(candidates, 1)

    #Finds where two different event's PDFs overlap: goes through matrix of allevents x allevents, 
    #marking '1' in the position of the matrix where two different events overlap in time (similar idea to adjacency matrix)
    for i in range(n_events):
        for j in range(n_events):
    #Skips diagonals and where an event overlaps with itself
            if i == j:
                continue

            if events.loc[i, "site"] == events.loc[j, "site"]:
                continue

            yr1 = marginal[
                (marginal["site"] == events.loc[i, "site"]) &
                (marginal["event"] == events.loc[i, "event"]) &
                (marginal["prob"] > threshold)
            ]["year"].values

            yr2 = marginal[
                (marginal["site"] == events.loc[j, "site"]) &
                (marginal["event"] == events.loc[j, "event"]) &
                (marginal["prob"] > threshold)
            ]["year"].values

            if (yr1.min() < yr2.max()) and (yr2.min() < yr1.max()):
                candidates[i, j] = 1
                candidates[j, i] = 1
    
    print("Candidates built...")

##Create the equivalent of site_binary_matrix but for the candidates matrix
    candidates_list = [[] for site in range(len(sites['site']) - 2)]
    for idx, trench in enumerate(sites['site'][1:-1]):
        indices = []
        for j in range(events.shape[0]):
            if events['site'][j] == trench:
                indices.append(j)
        for i in range(events.shape[0]):
           #Note: events from the same site can appear multiple times in a row if multiple events from another trench overlap them
        #Therefore, to convert to binary, make condition that if sum of indexed range is nonzero, position in candidates_binary is 1, else it is 0
            if (candidates[i][np.min(indices) : np.max(indices) + 1]).sum() > 0:
                candidates_list[idx].append(1)
            else:
                candidates_list[idx].append(0)

##Turn list of lists into matrix and transpose to get in same format as site_binary_matrix
    candidates_matrix = np.array(candidates_list)
    candidates_binary = candidates_matrix.T

#Make arrays of arrays (matrix) that just hold the indices of each trench site, in same order (to be used to figure out where to split each row of caniddates)
    site_indices = [[] for site in range(len(sites) - 2)]
    for idx, site in enumerate(sites['site'][1:-1]):
        site_indices[idx].append(np.where(events['site'] == site)[0])

##Code to find invalid gaps: restructure and use to filter candidates BEFORE creating all possible combinations

#Make new matrix to store the new valid rows created from candidates
#Make new matrix to store the new valid rows created from candidates
    new_candidates = []

    for i in range(len(candidates_binary)):
    #Ensure there are at least two 1s to have something "in-between"
        if np.count_nonzero(candidates_binary[i]) <= 1:
            new_candidates.append(candidates[i])
        else:
        #Find the boundary indexes of the first and last '1'
            all_ones = (np.where(candidates_binary[i] == 1))[0]
            first_one = all_ones[0]
            last_one = all_ones[-1]
        #Check if 0 exists inside this sliced sub-array
            middle = candidates_binary[i][first_one + 1 : last_one]
            if np.any(middle == 0):
        #Pasted from def valid_overlap: calculate the age range of the earthquake represented by this row
                cols = np.where(candidates[i] == 1)[0]
                selected = event_names[cols]
                dfs = [
                    marginal[
                        (marginal["event_name"] == e) &
                                (marginal["prob"] > threshold)
                            ]
                        for e in selected
                        ]
                mins = [df["year"].min() for df in dfs]
                maxs = [df["year"].max() for df in dfs]

                event_min = max(mins)
                event_max = min(maxs)

        #Find the youngest date of the age limits of the two records on either side by finding the youngest of the two oldest dates from each of the two record's PDFs
        ##Added + 1 to both first_one and last_one because marginal['site'] has length 10 and candidates_binary has length 8
                youngest_adjacent_record_base = max(min(marginal.loc[marginal['site'] == sites['site'][first_one + 1], 'year']), min(marginal.loc[marginal['site'] == sites['site'][last_one + 1], 'year']))
        #Find indexes of 0's in middle (to be used in next step to test if the gap of 0's is from only prerecord sites)
                ##IMPORTANT EDIT: Remove back of zeros_idx code
                zeros_idx = np.where(middle == 0)[0]# + (first_one)
##IMPORTANT EDIT: First one + 1 is needed because we're trying to find the actual index of the zero based on its index in the middle
        #(+1) and the index of the first element of the gap (first_one)
                zeros_idx = [x + (first_one + 1) for x in zeros_idx]
        #Make array of the oldest age of each site that contains zero in this middle gap
                min_ages = []
                for index in zeros_idx:
                    min_ages.append((marginal.loc[marginal['site'] == sites['site'][index + 1], 'year']).min())

                youngest_adjacent_record_base = []
                for index in zeros_idx:
                    ones_index = np.searchsorted(all_ones, index)
                    idx_below = all_ones[ones_index - 1]
                    idx_above = all_ones[ones_index]
                    adjacent_record_base = max(min(marginal.loc[marginal['site'] == sites['site'][idx_below + 1], 'year']), (min(marginal.loc[marginal['site'] == sites['site'][idx_above + 1], 'year']))) 
                    youngest_adjacent_record_base.append(adjacent_record_base)
        #Restructured from earlier version of code: if there is an invalid gap in this row, split up row at invalid gaps into seperate rows
        #Condition structured as 'if there is a zero in this middle gap AND the shortest of the two records on either side is still 
        #longer than the length of any of the records that are zero in this gap AND the age range of the earthquake represented by this row is older 
        #than the oldest part of any of the records that are 0 in this gap' then this is not valid for a single event and must be split up

        ##Plan: store the first and last indices of each site from events (same as original candidates), then use above condition to find
        #and store the indices of each gap, which can be matched back to index of trench, which can be matched back to the index to split the
        #row of candidates by, then add 0's back to rows, then replace original row with these split rows
                
                invalid_binary_gaps_indices = []
                for n in range(len(zeros_idx)):
                    if min_ages[n] < youngest_adjacent_record_base[n] or event_min > min_ages[n]:
                        invalid_binary_gaps_indices.append(zeros_idx[n])
        ##Added condition so that if there is a gap but it's a valid gap (leaving invalid_binary_gaps_indices empty), 
        #append the original candidates row and skip below
                if not invalid_binary_gaps_indices:
                    new_candidates.append(candidates[i])
                    continue
                split_indices = []
                for index in invalid_binary_gaps_indices:
                    split_indices.append(site_indices[index][0][0])
                for index in split_indices:
                    row = candidates[i]
                    row1 = row.copy()
                    row1[index:] = 0
                    row2 = row.copy()
                    row2[:index] = 0
                    new_candidates.append(row1)
                    new_candidates.append(row2)

###
            else:
                new_candidates.append(candidates[i])
###
    
    #Generates all possible combinations of events: finds the indices where each row contains 1s (idx), skips rows with no 1s, 
    all_combinations = []

    for i in range(n_events):
        idx = np.where(candidates[i] == 1)[0]
        k = len(idx)

        if k == 0:
            continue
        #Creates all possible combinations of 1s and 0s (except for the all zero combination, hence '-1')
        combos = as_binary_matrix(2**k - 1, k)
        #Initialize matrix: matrix of all 0, with columns being the events and rows the possible combinations
        mat = np.zeros((combos.shape[0], n_events), dtype=int)
        #Creates matrix holding all of the combinations that have 1s in the same position (index) specified by 'idx'
        mat[:, idx] = combos

        all_combinations.append(mat)
    #Combines all into one large matrix
    all_combinations = np.vstack(all_combinations)

    #Removes historical events
    historical = marginal[marginal["prob"] == 1]["event_name"].unique()
    #Specific to San Andreas Fault: Remove EL1 and Event_X (known to be 1857 and 1812 historical events, respectively)
    historical = np.append(historical, 'Elizabeth_Lake_Event_EL1')
    historical = np.append(historical, 'Pallett_Creek_Event_X')

    cols_keep = [i for i, e in enumerate(event_names) if e not in historical]

    event_keep_names = []
    for i in cols_keep:
        event_keep_names.append(event_names[i])
    #Remove the rows that turned into all zero rows after the removal of historical events
    drop_indices = []
    
    for i in range(len(all_combinations)):
        if sum(all_combinations[i]) == 0:
            drop_indices.append(i)

    all_combinations = np.delete(all_combinations, drop_indices, axis = 0)

    
    #Removes candidate events that include more than one event from a site
    sites_sorted = sites.sort_values("loc")
    site_names = sites_sorted["site"].values
    #Will store how many events from each site are in combination
    site_count = np.zeros((all_combinations.shape[0], len(site_names)))

    for i, site in enumerate(site_names):
        cols = [c for c in event_keep_names if c.startswith(site + "_")]
        idx = [np.where(event_names == c)[0][0] for c in cols]
        site_count[:, i] = all_combinations[:, idx].sum(axis=1)
    #Only keep events with <= 1 event per site
    mask = (site_count <= 1).all(axis=1)
    all_combinations = all_combinations[mask]
    site_count = site_count[mask]

    print("All combinations finished...")

    
## New new version to drop candidate events with gaps UNLESS that gap is from a site in the time before its record started

    drop_indices = []

    for i in range(len(all_combinations)):
    #Ensure there are at least two 1s to have something "in-between"
        if np.count_nonzero(site_count[i]) > 1:
        #Find the boundary indexes of the first and last '1'
            all_ones = (np.where(site_count[i] == 1))[0]
            first_one = all_ones[0]
            last_one = all_ones[-1]
        #Check if 0 exists inside this sliced sub-array
            middle = site_count[i][first_one + 1 : last_one]
            if np.any(middle == 0):
        ##Pasted from def valid_overlap: calculate the age range of the earthquake represented by this row
                cols = np.where(all_combinations[i] == 1)[0]
                selected = event_names[cols]
                dfs = [
                    marginal[
                        (marginal["event_name"] == e) &
                                (marginal["prob"] > threshold)
                            ]
                        for e in selected
                        ]
                mins = [df["year"].min() for df in dfs]
                maxs = [df["year"].max() for df in dfs]

                event_min = max(mins)
                event_max = min(maxs)
   
#Important note: the indexing for marginal is shifted down by one in this part of the code because site_count has the same dimension as marginal        
        #Find indexes of 0's in middle (to be used in next step to test if the gap of 0's is from only prerecord sites)
                zeros_idx = np.where(middle == 0)[0]
                zeros_idx = [x + (first_one + 1) for x in zeros_idx]
        #Make array of the oldest age of each site that contains zero in this middle gap
                min_ages = []
                for index in zeros_idx:
                    min_ages.append((marginal.loc[marginal['site'] == sites['site'][index], 'year']).min())
        #Find youngest of two records on sides of first and last zero: Will this break if there are two zero gaps 
####IMPORTANT EDIT: New version: find the indices of the 1's that bound each zero, and index marginal based on the indices of those 1's
#In other words, find the youngest adjacent record for each zero (to be used in next step)
                youngest_adjacent_record_base = []
                for index in zeros_idx:
                    ones_index = np.searchsorted(all_ones, index)
                    idx_below = all_ones[ones_index - 1]
                    idx_above = all_ones[ones_index]
                    adjacent_record_base = max(min(marginal.loc[marginal['site'] == sites['site'][idx_below], 'year']), (min(marginal.loc[marginal['site'] == sites['site'][idx_above], 'year']))) 
                    youngest_adjacent_record_base.append(adjacent_record_base)            
        #Condition structured as 'if there is a zero in this middle gap AND the shortest of the two records on either side is still 
        #longer than the length of any of the records that are zero in this gap AND the age range of the earthquake represented by this row is older 
        #than the oldest part of any of the records that are 0 in this gap' then this event is not added to drop_indices
                for n in range(len(zeros_idx)):
                    if min_ages[n] < youngest_adjacent_record_base[n] or event_min > min_ages[n]:
                        drop_indices.append(i)

###IMPORTANT EDIT: Added condition so that if there are multiple trench events from the same site in the same candidate event, add to drop_indices
        for j in range(len(site_indices)):
            if sum(all_combinations[i][site_indices[j][0][0]:site_indices[j][0][-1]]) > 1:
                drop_indices.append(i)
###
                    

    valid_combinations = np.delete(all_combinations, drop_indices, axis = 0)

##


    #Remove duplicate rows
    valid_combinations = np.unique(valid_combinations, axis=0)

    print("Valid combinations finished...")
    

    #Define function valid_overlap: find candidates where different trench events overlap in time
    def valid_overlap(row, threshold=0):
        cols = np.where(row == 1)[0]
        selected = event_names[cols]
    
        dfs = [
            marginal[
                (marginal["event_name"] == e) &
                    (marginal["prob"] > threshold)
            ]
            for e in selected
        ]

        if any(df.empty for df in dfs):
            return False

        mins = [df["year"].min() for df in dfs]
        maxs = [df["year"].max() for df in dfs]

        return max(mins) < min(maxs)


    #Use to find all candidate events whose constitive earthquake events have statistically significant overlap


## Original location for historical code through valid_combinations code
    mask = np.array([valid_overlap(r, threshold) for r in valid_combinations])
    valid_combinations = valid_combinations[mask]

    valid_combinations = valid_combinations[:, cols_keep]
#Remove rows of zeros from valid combinations
    valid_combinations = [row for row in valid_combinations if sum(row)>0]

    
    #Put valid_combinations into DataFrame to create 'candidates'
    candidates_df = pd.DataFrame(valid_combinations, columns=event_names[cols_keep])
    candidates_df["event_id"] = np.arange(1, len(candidates_df) + 1)

    print("Candidates dataframe finished, creating earthquake objects for each candidate...")
    

Enumeration_Option = 2

if Enumeration_Option == 2:
    #Create earthquake class
    class Earthquake:

        #Constructor: stores earthquake name, age, PDF, and CDF for each earthquake event (i.e. Bidart.Event_2)
        def __init__(self, name, age, PDF, CDF):
            self.name = name
            self.age = np.array(age)
            self.PDF = np.array(PDF)
            self.CDF = np.array(CDF)
        #Returns single sampled age based on PDF: generate random number between 0 and 1, then sample the age from the CDF that corresponds to number
        def event(self):
            r = np.random.rand()
            return self.age[np.argmin(np.abs(self.CDF - r))]
        #Returns n sampled ages: calls event() n times, then stores results in an array
        def events(self, n):
            return np.array([self.event() for _ in range(n)])
        #Resamples earthquake age to yearly resolution: calculates age range (L), then creates list of years inside age range and interpolates event PDF and CDF to those years
        def resample(self):
            L = int(self.age[-1] - self.age[0])
            new_age = np.linspace(self.age[0], self.age[-1], L + 1)

            f_age = interp1d(self.age, self.age, fill_value="extrapolate")
            f_pdf = interp1d(self.age, self.PDF, fill_value="extrapolate")
            f_cdf = interp1d(self.age, self.CDF, fill_value="extrapolate")

            self.age = f_age(new_age)
            self.PDF = f_pdf(new_age)
            self.PDF = self.PDF / np.sum(self.PDF)
            self.CDF = f_cdf(new_age)
        #Calculates joint probability of multiple earthquake events: find age spacing, and make sure both events have same spacing
        def jointprobability(self, other):
            diff_age = np.mean(np.diff(self.age))
            if diff_age != np.mean(np.diff(other.age)):
                raise ValueError("Sampling interval differs")
            #Calculate misallignment between both age groups, then use this to shift the two distributions so they line up
            misalign = (other.age[0] % diff_age) - (self.age[0] % diff_age)
            self_age = self.age.copy()
            other_age = other.age.copy()
            self_age = self_age + misalign
            #Find where distributions overlap
            lower = max(min(self_age), min(other_age))
            upper = min(max(self_age), max(other_age))

            if lower >= upper:
                return None
            #Create grid of shared ages
            new_age = np.arange(lower, upper + diff_age, diff_age)
            #Find starting and ending indexes for both event PDFs
            this_min = np.argmin(np.abs(self_age - lower))
            that_min = np.argmin(np.abs(other_age - lower))
            this_max = np.argmin(np.abs(self_age - upper))
            that_max = np.argmin(np.abs(other_age - upper))
            #Multiply PDFs element-wise
            new_pdf = self.PDF[this_min:this_max+1] * other.PDF[that_min:that_max+1]
            #Normalize (set sum of new PDF to 1)
            overlap = np.sum(new_pdf)
            new_pdf = new_pdf / overlap
            new_cdf = np.cumsum(new_pdf)

            return Earthquake(self.name, new_age, new_pdf, new_cdf)


    #Define function load_oxcal: Reads in events from each Oxcal file, pulls out and saves ages, pdf, and cdf data (outputs 'Earthquake' object)
    def load_oxcal(filename, name_selector='Event', delim=None):
        if delim:
            df = pd.read_csv(filename, sep=delim)
        else:
            df = pd.read_csv(filename)
    
        # Keep only posterior type
        df = df[df["type"] == "posterior"]
        df = df[df["name"].str.contains(name_selector)]

        result = []

        for eventname, group in df.groupby("name"):
            ages = group["value"].values
            pdf = group["probability"].values
            cdf = np.cumsum(pdf)
            sumcdf = cdf[-1]

            result.append(
                Earthquake(
                    name=eventname,
                    age=ages,
                    PDF=pdf / sumcdf,
                    CDF=cdf / sumcdf
                )
            )
        return result


        for eventname in events["name"].unique():
            event = events[events["name"] == eventname]


            ages = event["value"].values
            pdf = event["probability"].values
            cdf = np.cumsum(pdf)

            sumcdf = cdf[-1]
            result.append(
                Earthquake(
                    name=eventname,
                    age=ages,
                    PDF=pdf / sumcdf,
                    CDF=cdf / sumcdf
                )
            )
        return result


    #Define function get_marginal_pdfs: Builds dataframe with age, PDF, and name information as columns in dataframe with events as rows, 
    #also only keeps nonzero rows, and gives each event its own number
    def get_marginal_pdfs(oxcal):
        rows = []
        for i, eq in enumerate(oxcal):
            df = pd.DataFrame({
                "age": eq.age,
                "PDF": eq.PDF,
                "name": eq.name,
                ##
                "CDF": eq.CDF
                ##
            })
            df = df[df["age"] > 0]
            df["site_order"] = i
            rows.append(df)
        return rows

    #Define function as_binary_matrix: loops over dataframe to convert each row into binary
    def as_binary_matrix(n, width):
        return np.array([
            list(np.binary_repr(i, width=width))
            for i in range(1, n + 1)
        ], dtype=int)

##
#Original location for 'define sites dataframe'
##

    #Create marginal PDF dataframes for each paleoseismic site using load_oxcal function
    marginal_data = []

    for i, row in sites.iterrows():
        if pd.isna(row["file"]):
            df = pd.DataFrame({"age": [np.nan], "PDF": [np.nan], "name": [np.nan]})
        else:
            oxcal = load_oxcal(row["file"], row["event_code"])
            df = pd.concat(get_marginal_pdfs(oxcal), ignore_index=True)

        df["site"] = row["site"]
        marginal_data.append(df)

    sites["marginal_pdf"] = marginal_data

    #Convert marginal_pdf to list, dropping column and row names
    marginal = pd.concat(sites["marginal_pdf"].tolist(), ignore_index=True)
    marginal = marginal.dropna(subset=["name"])

    #Converts data type to float, exchanges 'age' and 'PDF' index with 'name' and 'event'
    marginal["year"] = marginal["age"].astype(float)
    marginal["prob"] = marginal["PDF"].astype(float)
    marginal = marginal.drop(columns=["age", "PDF"])
    marginal = marginal.rename(columns={"name": "event"}) 

    #Format each trench event to be labeled as site.event_# (i.e. Bidart.Event_2)
    marginal["event"] = marginal["event"].str.replace(r"\s+", "_", regex=True)
    marginal["event_name"] = marginal["site"] + "_" + marginal["event"]

    print("Marginal dataframe constructed...")

#Added sort=False because if groupby sorts alphabetically, indexing for jointprobability will be screwed up
    #Calculate mean year for each event (weighted by probability)
    events = (
        marginal.groupby(["site", "event"], sort=False)
        .apply(lambda x: np.average(x["year"], weights=x["prob"]))
        .reset_index(name="year")
    )

##Moved drop historical events code here and edited to work on events
    #Removes historical events
    historical = marginal[marginal["prob"] == 1]["event_name"].unique()
    #Specific to San Andreas Fault: Remove EL1 and Event_X (known to be 1857 and 1812 historical events, respectively)
    historical = np.append(historical, 'Elizabeth_Event_EL1')
    historical = np.append(historical, 'Pallett_Event_X')

    events_drop_indices = []
    for event in historical:
        events_drop_indices.append(events[(events['site'] == event.split('_')[0]) & (events['event'] == event.split('_', 1)[-1])].index[0])
    #ONLY ACTIVATE BELOW LINE WHEN ANNETTE IS INCLUDED IN MODEL: Specific to San Andreas fault: Drop index = 5, which is the historical Annette groundshaking event
    events_drop_indices.append(5)
    events.drop(events.index[events_drop_indices], inplace = True)

    #Reset indices after dropping historical events
    events.reset_index(drop=True, inplace=True)

    #To be used later: event count after deletion of historical events
    n_events = len(events)

##

    #Build candidates matrix, using similar formatting from above to label event_names, then initiliazing candidates as an identity matrix
    event_names = (events["site"] + "_" + events["event"]).values
    candidates = np.zeros((n_events, n_events), dtype=int)
    np.fill_diagonal(candidates, 1)

    #Finds where two different event's PDFs overlap: goes through matrix of allevents x allevents, 
    #marking '1' in the position of the matrix where two different events overlap in time (similar idea to adjacency matrix)
    for i in range(n_events):
        for j in range(n_events):
            if i == j:
                continue

            if events.loc[i, "site"] == events.loc[j, "site"]:
                continue

            yr1 = marginal[
                (marginal["site"] == events.loc[i, "site"]) &
                (marginal["event"] == events.loc[i, "event"]) &
                (marginal["CDF"] > threshold) &
                (marginal["CDF"] < 1 - threshold)
            ]["year"].values

            yr2 = marginal[
                (marginal["site"] == events.loc[j, "site"]) &
                (marginal["event"] == events.loc[j, "event"]) &
                (marginal["CDF"] > threshold) &
                (marginal["CDF"] < 1 - threshold)
            ]["year"].values

            if (yr1.min() < yr2.max()) and (yr2.min() < yr1.max()):
                candidates[i, j] = 1
                candidates[j, i] = 1

    print("Candidates built...")

##Create the equivalent of site_binary_matrix but for the candidates matrix
    candidates_list = [[] for site in range(len(sites['site']) - 2)]
    for idx, trench in enumerate(sites['site'][1:-1]):
        indices = []
        for j in range(events.shape[0]):
            if events['site'][j] == trench:
                indices.append(j)
        for i in range(events.shape[0]):
           #Note: events from the same site can appear multiple times in a row if multiple events from another trench overlap them
        #Therefore, to convert to binary, make condition that if sum of indexed range is nonzero, position in candidates_binary is 1, else it is 0
            if (candidates[i][np.min(indices) : np.max(indices) + 1]).sum() > 0:
                candidates_list[idx].append(1)
            else:
                candidates_list[idx].append(0)

##Turn list of lists into matrix and transpose to get in same format as site_binary_matrix
    candidates_matrix = np.array(candidates_list)
    candidates_binary = candidates_matrix.T

#Make arrays of arrays (matrix) that just hold the indices of each trench site, in same order (to be used to figure out where to split each row of caniddates)
    site_indices = [[] for site in range(len(sites) - 2)]
    for idx, site in enumerate(sites['site'][1:-1]):
        site_indices[idx].append(np.where(events['site'] == site)[0])

##Code to find invalid gaps: restructure and use to filter candidates BEFORE creating all possible combinations

#Make new matrix to store the new valid rows created from candidates
    new_candidates = []

    for i in range(len(candidates_binary)):
    #Ensure there are at least two 1s to have something "in-between"
        if np.count_nonzero(candidates_binary[i]) <= 1:
            new_candidates.append(candidates[i])
        else:
        #Find the boundary indexes of the first and last '1'
            all_ones = (np.where(candidates_binary[i] == 1))[0]
            first_one = all_ones[0]
            last_one = all_ones[-1]
        #Check if 0 exists inside this sliced sub-array
            middle = candidates_binary[i][first_one + 1 : last_one]
            if np.any(middle == 0):
        #Pasted from def valid_overlap: calculate the age range of the earthquake represented by this row
                cols = np.where(candidates[i] == 1)[0]
                selected = event_names[cols]
                dfs = [
                    marginal[
                        (marginal["event_name"] == e) &
                        (marginal["CDF"] > threshold) &
                        (marginal["CDF"] < 1 - threshold)
                            ]
                        for e in selected
                        ]
                mins = [df["year"].min() for df in dfs]
                maxs = [df["year"].max() for df in dfs]

                event_min = max(mins)
                event_max = min(maxs)

        #Find the youngest date of the age limits of the two records on either side by finding the youngest of the two oldest dates from each of the two record's PDFs
        ##Added + 1 to both first_one and last_one because marginal['site'] has length 10 and candidates_binary has length 8
                youngest_adjacent_record_base = max(min(marginal.loc[marginal['site'] == sites['site'][first_one + 1], 'year']), min(marginal.loc[marginal['site'] == sites['site'][last_one + 1], 'year']))
        #Find indexes of 0's in middle (to be used in next step to test if the gap of 0's is from only prerecord sites)
                ##IMPORTANT EDIT: Remove back of zeros_idx code
                zeros_idx = np.where(middle == 0)[0]# + (first_one)
##IMPORTANT EDIT: First one + 1 is needed because we're trying to find the actual index of the zero based on its index in the middle
        #(+1) and the index of the first element of the gap (first_one)
                zeros_idx = [x + (first_one + 1) for x in zeros_idx]
        #Make array of the oldest age of each site that contains zero in this middle gap
                min_ages = []
                for index in zeros_idx:
                    min_ages.append((marginal.loc[marginal['site'] == sites['site'][index + 1], 'year']).min())

                youngest_adjacent_record_base = []
                for index in zeros_idx:
                    ones_index = np.searchsorted(all_ones, index)
                    idx_below = all_ones[ones_index - 1]
                    idx_above = all_ones[ones_index]
                    adjacent_record_base = max(min(marginal.loc[marginal['site'] == sites['site'][idx_below + 1], 'year']), (min(marginal.loc[marginal['site'] == sites['site'][idx_above + 1], 'year']))) 
                    youngest_adjacent_record_base.append(adjacent_record_base)
        #Restructured from earlier version of code: if there is an invalid gap in this row, split up row at invalid gaps into seperate rows
        #Condition structured as 'if there is a zero in this middle gap AND the shortest of the two records on either side is still 
        #longer than the length of any of the records that are zero in this gap AND the age range of the earthquake represented by this row is older 
        #than the oldest part of any of the records that are 0 in this gap' then this is not valid for a single event and must be split up

        ##Plan: store the first and last indices of each site from events (same as original candidates), then use above condition to find
        #and store the indices of each gap, which can be matched back to index of trench, which can be matched back to the index to split the
        #row of candidates by, then add 0's back to rows, then replace original row with these split rows
                
                invalid_binary_gaps_indices = []
                for n in range(len(zeros_idx)):
                    if min_ages[n] < youngest_adjacent_record_base[n] or event_min > min_ages[n]:
                        invalid_binary_gaps_indices.append(zeros_idx[n])
        ##Added condition so that if there is a gap but it's a valid gap (leaving invalid_binary_gaps_indices empty), 
        #append the original candidates row and skip below
                if not invalid_binary_gaps_indices:
                    new_candidates.append(candidates[i])
                    continue
                split_indices = []
                for index in invalid_binary_gaps_indices:
                    split_indices.append(site_indices[index][0][0])
                for index in split_indices:
                    row = candidates[i]
                    row1 = row.copy()
                    row1[index:] = 0
                    row2 = row.copy()
                    row2[:index] = 0
                    new_candidates.append(row1)
                    new_candidates.append(row2)

###
            else:
                new_candidates.append(candidates[i])
###


    #Generates all possible combinations of events: finds the indices where each row contains 1s (idx), skips rows with no 1s, 
    all_combinations = []

    for row in new_candidates:
        one_indices = [i for i, val in enumerate(row) if val == 1]

        if len(one_indices) > (len(sites['site']) - 2):

    # Select groups of 1-indices equal to the maximum
    # number of trench events allowed
            for index_combination in itertools.combinations(
                one_indices, (len(sites['site']) - 2)
            ):

            # Generate all 0/1 combinations for those indices
                for combo in itertools.product(
                    [0, 1], repeat=len(index_combination)
                ):
    #Changed from variant = list(testrow) because if it is kept as a copy then there are rows with the original number of 1's
                    variant = np.zeros(len(row))

                # Change only the selected indices
                    for index, bit in zip(index_combination, combo):
                        variant[index] = bit

                    all_combinations.append(variant)

        else:

        # Generate all 0/1 combinations for all 1 positions
            for combo in itertools.product(
                [0, 1], repeat=len(one_indices)
            ):

                variant = list(row)

                for index, bit in zip(one_indices, combo):
                    variant[index] = bit

                all_combinations.append(variant)

###Old code:
    #for i in range(len(new_candidates)):
    #    idx = np.where(new_candidates[i] == 1)[0]
    #    k = len(idx)

    #    if k == 0:
    #        continue
    #    #Creates all possible combinations of 1s and 0s (except for the all zero combination, hence '-1')
    #    combos = as_binary_matrix(2**k - 1, k)
        #Initialize matrix: matrix of all 0, with columns being the events and rows the possible combinations
    #    mat = np.zeros((combos.shape[0], n_events), dtype=int)
        #Creates matrix holding all of the combinations that have 1s in the same position (index) specified by 'idx'
    #    mat[:, idx] = combos

    #    all_combinations.append(mat)
    #Combines all into one large matrix
    all_combinations = np.vstack(all_combinations)

    #Removes historical events
    historical = marginal[marginal["prob"] == 1]["event_name"].unique()
    #Specific to San Andreas Fault: Remove EL1 and Event_X (known to be 1857 and 1812 historical events, respectively)
    #Remove historical groundshaking event from Annette site
    historical = np.append(historical, 'Elizabeth_Lake_Event_EL1')
    historical = np.append(historical, 'Pallett_Creek_Event_X')
    historical = np.append(historical, 'Groundshaking_Event_')

    cols_keep = [i for i, e in enumerate(event_names) if e not in historical]

    event_keep_names = []
    for i in cols_keep:
        event_keep_names.append(event_names[i])
    #Remove the rows that turned into all zero rows after the removal of historical events
    all_combinations = [row for row in all_combinations if sum(row) != 0]

    #Convert all_combinations into a matrix, for next part of code

    all_combinations = np.array(all_combinations)
    
    #for i in range(len(all_combinations)):
    #    if sum(all_combinations[i]) == 0:
    #        all_combinations = np.delete(all_combinations, i, axis = 0)

    
    #Removes candidate events that include more than one event from a site
    sites_sorted = sites.sort_values("loc")
    site_names = sites_sorted["site"].values
    #Will store how many events from each site are in combination
    site_count = np.zeros((len(all_combinations), len(site_names)))

    for i, site in enumerate(site_names):
        cols = [c for c in event_keep_names if c.startswith(site + "_")]
        idx = [np.where(event_names == c)[0][0] for c in cols]
        site_count[:, i] = all_combinations[:, idx].sum(axis=1)

    #Only keep events with <= 1 event per site
    mask = (site_count <= 1).all(axis=1)
    all_combinations = all_combinations[mask]
    site_count = site_count[mask]

    print("All combinations finished...")

## New new version to drop candidate events with gaps UNLESS that gap is from a site in the time before its record started OR the gap can be explained by depositional hiatuses

    drop_indices = []

    for i in range(len(all_combinations)):
    #Ensure there are at least two 1s to have something "in-between"
        if np.count_nonzero(site_count[i]) > 1:
        #Find the boundary indexes of the first and last '1'
            all_ones = (np.where(site_count[i] == 1))[0]
            first_one = all_ones[0]
            last_one = all_ones[-1]
        #Check if 0 exists inside this sliced sub-array
            middle = site_count[i][first_one + 1 : last_one]
            if np.any(middle == 0):
        ##Pasted from def valid_overlap: calculate the age range of the earthquake represented by this row
                cols = np.where(all_combinations[i] == 1)[0]
                selected = event_names[cols]
                dfs = [
                    marginal[
                        (marginal["event_name"] == e) &
                        (marginal["CDF"] > threshold) &
                        (marginal["CDF"] < 1 - threshold)
                            ]
                        for e in selected
                        ]
                mins = [df["year"].min() for df in dfs]
                maxs = [df["year"].max() for df in dfs]

                event_min = max(mins)
                event_max = min(maxs)
   
#Important note: the indexing for marginal is shifted down by one in this part of the code because site_count has the same dimension as marginal        
        #Find indexes of 0's in middle (to be used in next step to test if the gap of 0's is from only prerecord sites)
                zeros_idx = np.where(middle == 0)[0]
                zeros_idx = [x + (first_one + 1) for x in zeros_idx]
        #Make array of the oldest age of each site that contains zero in this middle gap
                min_ages = []
                for index in zeros_idx:
                    min_ages.append((marginal.loc[marginal['site'] == sites['site'][index], 'year']).min())
        #Find youngest of two records on sides of first and last zero: Will this break if there are two zero gaps 
####IMPORTANT EDIT: New version: find the indices of the 1's that bound each zero, and index marginal based on the indices of those 1's
#In other words, find the youngest adjacent record for each zero (to be used in next step)
                youngest_adjacent_record_base = []
                for index in zeros_idx:
                    ones_index = np.searchsorted(all_ones, index)
                    idx_below = all_ones[ones_index - 1]
                    idx_above = all_ones[ones_index]
                    adjacent_record_base = max(min(marginal.loc[marginal['site'] == sites['site'][idx_below], 'year']), (min(marginal.loc[marginal['site'] == sites['site'][idx_above], 'year']))) 
                    youngest_adjacent_record_base.append(adjacent_record_base)            
        #Condition structured as 'if there is a zero in this middle gap AND the shortest of the two records on either side is still 
        #longer than the length of any of the records that are zero in this gap AND the age range of the earthquake represented by this row is older 
        #than the oldest part of any of the records that are 0 in this gap' then this event is not added to drop_indices
                for n in range(len(zeros_idx)):
                    if min_ages[n] < youngest_adjacent_record_base[n] or event_min > min_ages[n]: #If gap is invalid based on timing constraints, test for depositional hiatuses to explain gap before adding to drop_indices
                        trench = trench_names[zeros_idx[n]]
                        test = Sedimentary_Hiatus_df.index[(Sedimentary_Hiatus_df['trench'] == trench) & (Sedimentary_Hiatus_df['upper age'] > event_max) & (Sedimentary_Hiatus_df['lower age'] < event_min)]
                        if len(test) == 0:
                            drop_indices.append[i]

###IMPORTANT EDIT: Added condition so that if there are multiple trench events from the same site in the same candidate event, add to drop_indices
        for j in range(len(site_indices)):
            if sum(all_combinations[i][site_indices[j][0][0]:site_indices[j][0][-1]]) > 1:
                drop_indices.append(i)
###
                    

    valid_combinations = np.delete(all_combinations, drop_indices, axis = 0)

##

    #Remove duplicate rows
    valid_combinations = np.unique(valid_combinations, axis=0)

    print("Valid combinations finished...")

    #Define function valid_overlap: find years where different trench events overlap
    def valid_overlap(row, threshold=threshold):
        cols = np.where(row == 1)[0]
        selected = event_names[cols]
    
        dfs = [
            marginal[
                (marginal["event_name"] == e) &
                    (marginal["CDF"] > threshold) &
                    (marginal["CDF"] < 1 - threshold)
            ]
            for e in selected
        ]

        if any(df.empty for df in dfs):
            return False

        mins = [df["year"].min() for df in dfs]
        maxs = [df["year"].max() for df in dfs]

        return max(mins) < min(maxs)


    #Use to find all candidate events whose consitutive earthquake events have statistically significant overlap


## Original location for historical code through valid_combinations code
    mask = np.array([valid_overlap(r, threshold) for r in valid_combinations])
    valid_combinations = valid_combinations[mask]

    valid_combinations = valid_combinations[:, cols_keep]
#Remove rows of zeros from valid combinations
    valid_combinations = [row for row in valid_combinations if sum(row)>0]

    
    #Put valid_combinations into DataFrame to create 'candidates'
    candidates_df = pd.DataFrame(valid_combinations, columns=event_names[cols_keep])
    candidates_df["event_id"] = np.arange(1, len(candidates_df) + 1)

    print("Candidates dataframe finished, creating earthquake objects for each candidate...")

all_oxcal = []

for i, row in sites.iterrows():
    if not pd.isna(row["file"]):
        oxcal = load_oxcal(row["file"], row["event_code"])
        all_oxcal.extend(oxcal)

#Stores 55 events due to 6 historic events + duplicated (14) Wrightwood events
#Delete historical events from all_oxcal (using the fact that their ages will be whole-number years)
for n in reversed(range(len(all_oxcal))):
    all_whole = all(x % 1 == 0 for x in all_oxcal[n].age)
    if all_whole == True:
        del all_oxcal[n]

#Specific for San Andreas Fault since there are extra Wrightwood events in csv for some reason (28 instead of 14 with names like 'Event W510')
all_oxcal = [n for n in all_oxcal if len(n.name) < 10]

#Specific for San Andreas Fault: remove Pallett Event X and Elizabeth Lake Event EL1 from all_oxcal since it is now known as the historical 1812 earthquake
for i in range(len(all_oxcal)-2):
    if np.mean(all_oxcal[i].age) == 1748.0 and all_oxcal[i].name == 'Event X' or all_oxcal[i].name == 'Event EL1':
        all_oxcal.pop(i)

candidate_earthquakes = {}
for i in range(candidates_df.shape[0]):
    row = candidates_df.iloc[i,:-1].to_numpy()
    ones_indices_ce = np.nonzero(row)[0]
    joint = all_oxcal[ones_indices_ce[0]]
    if np.sum(row) == 1:
        key = f"individual{i}"
        candidate_earthquakes[key] = joint
    if np.sum(row) >= 2:
        for idx in ones_indices_ce[1:]:
            joint = joint.jointprobability(all_oxcal[idx])
            key = f"joint{i}"
            candidate_earthquakes[key] = joint



candidate_earthquake_ages = np.zeros(len(list(candidate_earthquakes)))
for i in range(len(list(candidate_earthquakes))):
    name = list(candidate_earthquakes)[i]
    candidate_earthquake_ages[i] = np.average(candidate_earthquakes[name].age, weights=candidate_earthquakes[name].PDF)

#Download candidates dataframe for next part of notebook:
#candidates_df.to_csv("C:/Users/erikc/Downloads/Candidates.csv") 